In [ ]:
# ===========================
# 0) DATASET EXTRACT (Drive -> local -> /content)
# - Fixes occasional Colab/Drive Errno 107 (Transport endpoint is not connected)
# - Copies ZIPs from Drive to /content first, then extracts locally with streaming.
# ===========================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
from zipfile import ZipFile
import shutil, time

# Source ZIPs in Drive
DRIVE = Path("/content/drive/MyDrive/deepfake_project")
ZIP_TRAIN_DRIVE = DRIVE / "zips/dataset_train.zip"
ZIP_TEST_DRIVE  = DRIVE / "zips/dataset_test.zip"

# Local copies (avoid streaming directly from Drive during unzip)
ZIP_TRAIN_LOCAL = Path("/content/dataset_train.zip")
ZIP_TEST_LOCAL  = Path("/content/dataset_test.zip")

# Destination folders in Colab
DST_AFHQ_TRAIN = Path("/content/stargan-v2/combined_balanced/train")
DST_AFHQ_TEST  = Path("/content/stargan-v2/combined_balanced/test")

DST_AFHQ_TRAIN.mkdir(parents=True, exist_ok=True)
DST_AFHQ_TEST.mkdir(parents=True, exist_ok=True)

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

def copy_to_local(src: Path, dst: Path):
    if not src.exists():
        raise FileNotFoundError(f"ZIP not found: {src}")
    print(f"Copying ZIP to local: {src} -> {dst}")
    shutil.copy2(src, dst)
    print(f"Local ZIP size (MB): {dst.stat().st_size / (1024*1024):.2f}")

def safe_extract_images_stream(zip_path: Path, dest_dir: Path):
    """Extract only image files from zip_path to dest_dir (preserves internal subpaths),
    with path traversal protection and streaming copy (memory-safe)."""
    extracted = 0
    with ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            suffix = Path(info.filename).suffix.lower()
            if suffix not in IMG_EXT:
                continue

            # Path traversal protection
            target_path = dest_dir / info.filename
            target_path_parent = target_path.parent.resolve()
            dest_root_resolved = dest_dir.resolve()
            if not str(target_path_parent).startswith(str(dest_root_resolved)):
                continue

            target_path.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info, "r") as src_f, open(target_path, "wb") as dst_f:
                shutil.copyfileobj(src_f, dst_f, length=1024*1024)  # 1MB chunks
            extracted += 1
    return extracted

def extract_with_retry(zip_local: Path, dst: Path, label: str, retries=1):
    for attempt in range(retries + 1):
        try:
            n = safe_extract_images_stream(zip_local, dst)
            print(f"[DONE] Extracted {label} -> {dst}: {n} files")
            return n
        except OSError as e:
            print(f"[WARN] {label} extraction error: {e}")
            if attempt < retries:
                print("[INFO] Retrying after Drive remount…")
                time.sleep(2)
                drive.mount('/content/drive', force_remount=True)
                time.sleep(2)
            else:
                raise

# Copy ZIPs locally first (key reliability step)
copy_to_local(ZIP_TRAIN_DRIVE, ZIP_TRAIN_LOCAL)
copy_to_local(ZIP_TEST_DRIVE,  ZIP_TEST_LOCAL)

# Extract from LOCAL ZIPs
n_train = extract_with_retry(ZIP_TRAIN_LOCAL, DST_AFHQ_TRAIN, "TRAIN", retries=1)
n_test  = extract_with_retry(ZIP_TEST_LOCAL,  DST_AFHQ_TEST,  "TEST",  retries=1)

print(f"[SUMMARY] TRAIN extracted: {n_train} | TEST extracted: {n_test}")


Mounted at /content/drive
Copying ZIP to local: /content/drive/MyDrive/deepfake_project/zips/dataset_train.zip -> /content/dataset_train.zip
Local ZIP size (MB): 650.01
Copying ZIP to local: /content/drive/MyDrive/deepfake_project/zips/dataset_test.zip -> /content/dataset_test.zip
Local ZIP size (MB): 276.58
[DONE] Extracted TRAIN -> /content/stargan-v2/combined_balanced/train: 22572 files
[DONE] Extracted TEST -> /content/stargan-v2/combined_balanced/test: 9672 files
[SUMMARY] TRAIN extracted: 22572 | TEST extracted: 9672


In [ ]:
!pip install shap lime --quiet

import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt
import numpy as np
import os

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
EXPLAIN_DIR = "/content/stargan-v2/explainability"
os.makedirs(EXPLAIN_DIR, exist_ok=True)

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
import xgboost as xgb

def _predict_proba_xgb_any(model, X):
    """
    Return P(y=1) for either:
      - xgboost.Booster (native API)
      - sklearn-style XGBClassifier (predict_proba)
    """
    X = np.asarray(X)
    # Native booster
    if isinstance(model, xgb.Booster):
        d = xgb.DMatrix(X)
        proba = model.predict(d, iteration_range=(0, model.best_iteration + 1))
        return np.asarray(proba).reshape(-1)
    # sklearn wrapper
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    # fallback
    raise TypeError(f"Unsupported model type: {type(model)}")

def explain_xgb_with_shap(
    model,
    X_train,
    X_test,
    y_test,
    feature_names,
    prefix="xgb",
    out_dir=None,
    max_bg=2000,
    max_test=2000,
    random_state=42,
    local_examples=True
):
    """
    SHAP explainability that works for BOTH Booster + XGBClassifier.

    Saves:
      - beeswarm summary plot
      - bar importance plot
      - importance CSV
      - (optional) a few local waterfall plots (TP/FP/FN if available)
    Returns:
      dict with paths + importance df
    """
    print(f"=== SHAP explainability for {prefix} ===")

    X_train_np = np.asarray(X_train)
    X_test_np  = np.asarray(X_test)
    y_test_np  = np.asarray(y_test).astype(int)

    # Output directory
    if out_dir is None:
        out_dir = Path.cwd() / "explainability"
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    rng = np.random.RandomState(random_state)

    # Background sample for speed
    if len(X_train_np) > max_bg:
        bg_idx = rng.choice(len(X_train_np), size=max_bg, replace=False)
        X_bg = X_train_np[bg_idx]
    else:
        X_bg = X_train_np

    # Test sample for speed
    if len(X_test_np) > max_test:
        te_idx = rng.choice(len(X_test_np), size=max_test, replace=False)
        X_te = X_test_np[te_idx]
        y_te = y_test_np[te_idx]
        te_idx_global = te_idx
    else:
        X_te = X_test_np
        y_te = y_test_np
        te_idx_global = np.arange(len(X_test_np))

    # Build explainer (TreeExplainer supports xgb.Booster directly)
    # Use feature_perturbation="tree_path_dependent" for tree models (default is fine too).
    explainer = shap.TreeExplainer(model, data=X_bg)

    shap_values = explainer.shap_values(X_te)  # shape: (n, p) for binary
    # For some versions, shap_values can be list-like; normalize:
    if isinstance(shap_values, list):
        # binary classifiers sometimes return [class0, class1]; use class 1
        shap_values = shap_values[1]

    # Summary (beeswarm)
    bees_path = out_dir / f"{prefix}_shap_summary_beeswarm.png"
    plt.figure()
    shap.summary_plot(shap_values, X_te, feature_names=feature_names, show=False)
    plt.tight_layout()
    plt.savefig(bees_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  - Saved SHAP beeswarm to {bees_path}")

    # Bar importance
    bar_path = out_dir / f"{prefix}_shap_importance_bar.png"
    plt.figure()
    shap.summary_plot(shap_values, X_te, feature_names=feature_names, plot_type="bar", show=False)
    plt.tight_layout()
    plt.savefig(bar_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  - Saved SHAP bar plot to {bar_path}")

    # Importance CSV
    mean_abs = np.mean(np.abs(shap_values), axis=0)
    imp_df = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": mean_abs
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    csv_path = out_dir / f"{prefix}_shap_importance.csv"
    imp_df.to_csv(csv_path, index=False)
    print(f"  - Saved SHAP importance CSV to {csv_path}")

    results = {
        "beeswarm_path": str(bees_path),
        "bar_path": str(bar_path),
        "importance_csv": str(csv_path),
        "importance_df": imp_df
    }

    # Optional local explanations (TP / FP / FN)
    if local_examples:
        proba_all = _predict_proba_xgb_any(model, X_test_np)
        pred_all = (proba_all >= 0.5).astype(int)

        # find indices in full test set
        tp = np.where((pred_all == 1) & (y_test_np == 1))[0]
        fp = np.where((pred_all == 1) & (y_test_np == 0))[0]
        fn = np.where((pred_all == 0) & (y_test_np == 1))[0]

        picks = []
        if len(tp) > 0: picks.append(("TP", int(tp[0])))
        if len(fp) > 0: picks.append(("FP", int(fp[0])))
        if len(fn) > 0: picks.append(("FN", int(fn[0])))

        for tag, idx in picks:
            x_row = X_test_np[idx:idx+1]
            # local shap values
            sv_row = explainer.shap_values(x_row)
            if isinstance(sv_row, list):
                sv_row = sv_row[1]

            # expected value handling (can be scalar or array-like)
            base = explainer.expected_value
            if isinstance(base, (list, np.ndarray)) and len(np.atleast_1d(base)) > 1:
                base = np.atleast_1d(base)[1]
            base = float(np.atleast_1d(base)[0])

            # Waterfall plot (new API)
            w_path = out_dir / f"{prefix}_shap_local_{tag}_idx{idx}.png"
            plt.figure()
            shap.plots.waterfall(
                shap.Explanation(
                    values=sv_row[0],
                    base_values=base,
                    data=x_row[0],
                    feature_names=feature_names
                ),
                show=False,
                max_display=min(12, len(feature_names))
            )
            plt.tight_layout()
            plt.savefig(w_path, dpi=200, bbox_inches="tight")
            plt.close()

            print(f"  - Saved local {tag} waterfall to {w_path}")

        results["local_examples"] = [f"{prefix}_shap_local_{tag}_idx{idx}.png" for tag, idx in picks]

    return results

In [ ]:
ZIP_BOVW = DRIVE / "output/bovw3/stargan-v2_data_bovw_kmeans_pca-20260308-230713.zip"
DST_BOVW   = Path("/content/stargan-v2/bovw")
DST_BOVW.mkdir(parents=True, exist_ok=True)

def safe_extract_files(zip_path: Path, dest_dir: Path):
    """
    Extract only image files from the zip to dest_dir (preserves internal subpaths),
    with path traversal protection.
    """
    if not zip_path.exists():
        raise FileNotFoundError(f"ZIP not found: {zip_path}")

    extracted = 0
    with ZipFile(zip_path, "r") as zf:
        for info in zf.infolist():
            # Skip directories
            if info.is_dir():
                continue


            # Path traversal protection
            target_path = dest_dir / info.filename
            target_path_parent = target_path.parent.resolve()
            dest_root_resolved = dest_dir.resolve()
            if not str(target_path_parent).startswith(str(dest_root_resolved)):
                # Skip anything that tries to escape the destination
                continue

            # Make sure subdirs exist
            target_path.parent.mkdir(parents=True, exist_ok=True)

            # Extract this single file
            with zf.open(info, "r") as src_f, open(target_path, "wb") as dst_f:
                dst_f.write(src_f.read())
            extracted += 1
    return extracted

n_bovw = safe_extract_files(ZIP_BOVW, DST_BOVW)
print(f"[DONE] Extracted BOVW files -> {DST_BOVW}: {n_bovw} files")

[DONE] Extracted BOVW files -> /content/stargan-v2/bovw: 22609 files


In [ ]:
!pip install -q xgboost

import time
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_recall_fscore_support
)
from xgboost import XGBClassifier

In [ ]:
BOVW_DIR = Path("/content/stargan-v2/bovw/bovw")

# dtype fix: fake_technique column contains mixed types (str + float NaN for real images)
# Specifying str dtype suppresses the DtypeWarning and ensures consistent string comparisons.
train_df = pd.read_csv(BOVW_DIR / "image_features.csv",      dtype={"fake_technique": str})
test_df  = pd.read_csv(BOVW_DIR / "image_features_test.csv", dtype={"fake_technique": str})

print(train_df.shape, test_df.shape)
train_df.head()


(22572, 439) (9672, 439)


,row_id,split,path,relative_path,filename,label,label_name,species,fake_technique,width,...,pca_86,pca_87,pca_88,pca_89,pca_90,pca_91,pca_92,pca_93,pca_94,pca_95
0,0,train,/content/stargan-v2/combined_balanced/train/fa...,fake/copy_move/copy_move__00329a52__pixabay_do...,copy_move__00329a52__pixabay_dog_003051_copymo...,1,fake,dog,copy_move,256,...,-0.013822,-0.020966,-0.007332,0.025829,0.020360,-0.024048,0.008494,-0.002867,0.003608,-0.038493
1,1,train,/content/stargan-v2/combined_balanced/train/fa...,fake/copy_move/copy_move__004af61d__pixabay_do...,copy_move__004af61d__pixabay_dog_003870_copymo...,1,fake,dog,copy_move,256,...,-0.003767,0.039712,0.026920,-0.005130,-0.003551,-0.019378,0.013667,0.022576,0.018257,0.031552
2,2,train,/content/stargan-v2/combined_balanced/train/fa...,fake/copy_move/copy_move__008a6e40__flickr_wil...,copy_move__008a6e40__flickr_wild_002771_copymo...,1,fake,wild,copy_move,256,...,-0.004998,-0.002633,-0.031530,-0.008713,-0.002245,-0.023739,0.001776,0.005949,0.009417,0.008716
3,3,train,/content/stargan-v2/combined_balanced/train/fa...,fake/copy_move/copy_move__00ae54fc__pixabay_ca...,copy_move__00ae54fc__pixabay_cat_000970_copymo...,1,fake,cat,copy_move,256,...,0.023029,0.023744,-0.012659,-0.010610,-0.017354,-0.016202,0.021908,-0.029520,0.002049,0.003073
4,4,train,/content/stargan-v2/combined_balanced/train/fa...,fake/copy_move/copy_move__00ef8c40__flickr_dog...,copy_move__00ef8c40__flickr_dog_000605_copymov...,1,fake,dog,copy_move,256,...,0.004637,0.007739,0.046452,0.018279,0.027167,-0.009621,0.022425,-0.006573,-0.041770,0.048629


In [ ]:
# ===========================
# Ordering + label alignment (CRITICAL)
# - We must guarantee that:
#   (a) train_df/test_df rows are in a deterministic filename order
#   (b) image path lists are sorted by filename
#   (c) all prediction arrays are keyed/merged by filename before stacking (ensemble)
# ===========================
from pathlib import Path

def _detect_path_col(df: pd.DataFrame):
    for c in ["filename","file","image","img","img_file","img_path","path","filepath","full_path"]:
        if c in df.columns:
            return c
    return None

def _add_filename_col(df: pd.DataFrame, out_col="filename"):
    col = _detect_path_col(df)
    if col is None:
        raise ValueError(
            "Could not find a filename/path column in df. "
            "Expected one of: filename, file, image, img_path, path, filepath, full_path."
        )
    df = df.copy()
    df[out_col] = df[col].astype(str).map(lambda x: Path(x).name)
    return df

# Add a canonical 'filename' column and sort deterministically
train_df = _add_filename_col(train_df, "filename").sort_values("filename").reset_index(drop=True)
test_df  = _add_filename_col(test_df,  "filename").sort_values("filename").reset_index(drop=True)

# Labels from tabular df (DO NOT overwrite later)
y_train = train_df["label"].astype(int).values
y_test_df = test_df["label"].astype(int).values  # keep as df-derived ground truth


In [ ]:
# y_train and y_test are defined in the alignment cell above.
# Keep y_test as the df-derived ground truth (y_test_df) for tabular models.
y_test = y_test_df


In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

def _binary_metrics(y_true, proba, thr=0.5):
    y_true = np.asarray(y_true).astype(int)
    proba  = np.asarray(proba).astype(float)
    pred   = (proba >= thr).astype(int)

    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())

    eps = 1e-12
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    prec = tp / max(tp + fp, 1)
    rec  = tp / max(tp + fn, 1)
    spec = tn / max(tn + fp, 1)
    f1   = (2 * prec * rec) / max(prec + rec, eps)

    auroc = roc_auc_score(y_true, proba)
    ap    = average_precision_score(y_true, proba)

    return {
        "threshold": float(thr),
        "AUROC": float(auroc),
        "AP": float(ap),
        "Accuracy": float(acc),
        "Precision": float(prec),
        "Recall": float(rec),
        "Specificity": float(spec),
        "F1": float(f1),
        "TP": tp, "TN": tn, "FP": fp, "FN": fn
    }

def run_xgb_cv_and_test(
    X_train, y_train,
    X_test,  y_test,
    model_name="XGB",
    n_splits=5,
    seed=42,
    val_size=0.2,
    early_rounds=50,
    thr=0.5,
    params=None,
    num_boost_round=5000
):
    """
    Version-safe XGBoost training using xgboost.train() + DMatrix.
    Supports early stopping via early_stopping_rounds (native API).
    Returns:
      booster_final, oof_proba, test_proba, cv_summary(dict), test_metrics(dict)
    """
    X_train = np.asarray(X_train)
    y_train = np.asarray(y_train).astype(int)
    X_test  = np.asarray(X_test)
    y_test  = np.asarray(y_test).astype(int)

    if params is None:
        params = {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "eta": 0.01,
            "max_depth": 4,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "lambda": 1.0,
            "min_child_weight": 1.0,
            "seed": seed,
            "verbosity": 0,
        }

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.full(len(y_train), np.nan, dtype=float)
    fold_aucs, fold_aps = [], []

    print(f"=== {model_name}: {n_splits}-fold CV (native early stopping) ===")

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, y_train), start=1):
        dtr = xgb.DMatrix(X_train[tr_idx], label=y_train[tr_idx])
        dva = xgb.DMatrix(X_train[va_idx], label=y_train[va_idx])

        booster = xgb.train(
            params=params,
            dtrain=dtr,
            num_boost_round=num_boost_round,
            evals=[(dva, "val")],
            early_stopping_rounds=early_rounds,
            verbose_eval=False
        )

        proba_va = booster.predict(dva, iteration_range=(0, booster.best_iteration + 1))
        oof[va_idx] = proba_va

        auc = roc_auc_score(y_train[va_idx], proba_va)
        ap  = average_precision_score(y_train[va_idx], proba_va)
        fold_aucs.append(auc)
        fold_aps.append(ap)

        print(f"Fold {fold}: AUROC={auc:.4f}, AP={ap:.4f}, best_iter={booster.best_iteration}")

    cv_summary = {
        "model": model_name,
        "n_splits": n_splits,
        "AUROC_mean": float(np.mean(fold_aucs)),
        "AUROC_std":  float(np.std(fold_aucs, ddof=1)) if len(fold_aucs) > 1 else 0.0,
        "AP_mean": float(np.mean(fold_aps)),
        "AP_std":  float(np.std(fold_aps, ddof=1)) if len(fold_aps) > 1 else 0.0
    }

    # Final train with a held-out validation split + early stopping
    X_trf, X_vaf, y_trf, y_vaf = train_test_split(
        X_train, y_train, test_size=val_size, stratify=y_train, random_state=seed
    )
    dtrf = xgb.DMatrix(X_trf, label=y_trf)
    dvaf = xgb.DMatrix(X_vaf, label=y_vaf)
    dte  = xgb.DMatrix(X_test, label=y_test)

    booster_final = xgb.train(
        params=params,
        dtrain=dtrf,
        num_boost_round=num_boost_round,
        evals=[(dvaf, "val")],
        early_stopping_rounds=early_rounds,
        verbose_eval=False
    )

    test_proba = booster_final.predict(dte, iteration_range=(0, booster_final.best_iteration + 1))
    test_metrics = _binary_metrics(y_test, test_proba, thr=thr)

    return booster_final, oof, test_proba, cv_summary, test_metrics

In [ ]:
# RQ1: Forensic feature model (file_size EXCLUDED — confirmed confound via lift analysis)
# file_size perfectly separates StarGAN tiles from other images due to generation artifacts,
# not genuine forensic signal. See RQ1a (cell below) for the sensitivity analysis with file_size.
forensic_cols = [
    "brightness", "contrast", "sharpness_l1_mean",
    "edge_density", "lap_var",
    # "file_size"  # EXCLUDED: lift table confirmed this is a generation artifact, not forensic signal
]

X_train_forensic = train_df[forensic_cols].values
X_test_forensic  = test_df[forensic_cols].values

forensic_model, forensic_oof, forensic_test_proba, forensic_cv, forensic_test = \
    run_xgb_cv_and_test(X_train_forensic, y_train,
                        X_test_forensic, y_test,
                        model_name="RQ1_Forensic_XGB")

print("\nRQ1 forensic TEST metrics:", forensic_test)
print("CV summary:", forensic_cv)

import xgboost as xgb
xgb.__version__


=== RQ1_Forensic_XGB: 5-fold CV (native early stopping) ===
Fold 1: AUROC=0.6379, AP=0.6743, best_iter=574
Fold 2: AUROC=0.6291, AP=0.6724, best_iter=594
Fold 3: AUROC=0.6269, AP=0.6669, best_iter=558
Fold 4: AUROC=0.6289, AP=0.6746, best_iter=706
Fold 5: AUROC=0.6315, AP=0.6735, best_iter=587

RQ1 forensic TEST metrics: {'threshold': 0.5, 'AUROC': 0.6386483909621866, 'AP': 0.670664263941583, 'Accuracy': 0.5988420181968569, 'Precision': 0.6449363250454822, 'Recall': 0.43982630272952855, 'Specificity': 0.7578577336641853, 'F1': 0.52298991885911, 'TP': 2127, 'TN': 3665, 'FP': 1171, 'FN': 2709}
CV summary: {'model': 'RQ1_Forensic_XGB', 'n_splits': 5, 'AUROC_mean': 0.6308426353821092, 'AUROC_std': 0.004259737115550548, 'AP_mean': 0.67234553116176, 'AP_std': 0.0031455491371993323}


'3.2.0'

In [ ]:
# Example: RQ1 forensic model
# Assume:
#   xgb_forensic_full : fitted XGB model
#   X_forensic_train, X_forensic_test
#   y_train, y_test
#   forensic_cols: list of forensic feature column names

shap_rq1 = explain_xgb_with_shap(
    model=forensic_model,
    X_train=X_train_forensic,
    X_test=X_test_forensic,
    y_test=y_test,
    feature_names=forensic_cols,
    prefix="rq1_forensic_xgb"
)

=== SHAP explainability for rq1_forensic_xgb ===


 98%|===================| 1954/2000 [00:24<00:00]       

  - Saved SHAP beeswarm to /content/explainability/rq1_forensic_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/explainability/rq1_forensic_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/explainability/rq1_forensic_xgb_shap_importance.csv
  - Saved local TP waterfall to /content/explainability/rq1_forensic_xgb_shap_local_TP_idx1705.png
  - Saved local FP waterfall to /content/explainability/rq1_forensic_xgb_shap_local_FP_idx9.png
  - Saved local FN waterfall to /content/explainability/rq1_forensic_xgb_shap_local_FN_idx1696.png


In [ ]:
def require_cols(df, cols, context=""):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"[{context}] Missing {len(missing)} required columns. "
            f"Example missing: {missing[:10]}"
        )

def require_any_prefix(df, prefixes, context=""):
    found = []
    for p in prefixes:
        found.extend([c for c in df.columns if c.startswith(p)])
    if len(found) == 0:
        raise ValueError(
            f"[{context}] Found 0 columns with prefixes {prefixes}. "
            f"This usually means Program 3 features were not merged into this dataframe."
        )
    return found

# Guard for BoVW/KMeans feature availability
bovw_cols = [c for c in train_df.columns if c.startswith("bovw_")]
prob_k2_cols  = [c for c in train_df.columns if c.startswith("prob_k2_")]
prob_k10_cols = [c for c in train_df.columns if c.startswith("prob_k10_")]
prob_k50_cols = [c for c in train_df.columns if c.startswith("prob_k50_")]

print("BoVW dims:", len(bovw_cols))
print("prob_k2 dims:", len(prob_k2_cols), "prob_k10 dims:", len(prob_k10_cols), "prob_k50 dims:", len(prob_k50_cols))

_ = require_any_prefix(train_df, ["bovw_", "prob_k2_", "prob_k10_", "prob_k50_"], context="RQ2/RQ3 features check")

BoVW dims: 257
prob_k2 dims: 2 prob_k10 dims: 10 prob_k50 dims: 50


In [ ]:
# grab all BoVW columns
bovw_cols = [c for c in train_df.columns if c.startswith("bovw_")]
print("BoVW dims:", len(bovw_cols))

# soft cluster probabilities (if present)
prob_k2_cols  = [c for c in train_df.columns if c.startswith("prob_k2_")]
prob_k10_cols = [c for c in train_df.columns if c.startswith("prob_k10_")]
prob_k50_cols = [c for c in train_df.columns if c.startswith("prob_k50_")]

print("prob_k2 dims:", len(prob_k2_cols),
      "prob_k10 dims:", len(prob_k10_cols),
      "prob_k50 dims:", len(prob_k50_cols))

bovw_k_feats = bovw_cols + prob_k2_cols + prob_k10_cols + prob_k50_cols

X_train_bovw_k = train_df[bovw_k_feats].values
X_test_bovw_k  = test_df[bovw_k_feats].values

bovw_k_model, bovw_k_oof, bovw_k_test_proba, bovw_k_cv, bovw_k_test = \
    run_xgb_cv_and_test(X_train_bovw_k, y_train,
                        X_test_bovw_k, y_test,
                        model_name="RQ2_BoVW_plus_Kmeans_XGB")

print("\nRQ2 BoVW+K TEST metrics:", bovw_k_test)


BoVW dims: 257
prob_k2 dims: 2 prob_k10 dims: 10 prob_k50 dims: 50
=== RQ2_BoVW_plus_Kmeans_XGB: 5-fold CV (native early stopping) ===
Fold 1: AUROC=0.5597, AP=0.5509, best_iter=266
Fold 2: AUROC=0.5680, AP=0.5559, best_iter=214
Fold 3: AUROC=0.5679, AP=0.5536, best_iter=213
Fold 4: AUROC=0.5741, AP=0.5662, best_iter=407
Fold 5: AUROC=0.5675, AP=0.5547, best_iter=420

RQ2 BoVW+K TEST metrics: {'threshold': 0.5, 'AUROC': 0.5884488903529566, 'AP': 0.5826067978474505, 'Accuracy': 0.5648263027295285, 'Precision': 0.561266367011921, 'Recall': 0.5938792390405294, 'Specificity': 0.5357733664185277, 'F1': 0.5771124284135436, 'TP': 2872, 'TN': 2591, 'FP': 2245, 'FN': 1964}


In [ ]:
# RQ2: BoVW+KMeans model
shap_rq2 = explain_xgb_with_shap(
    model=bovw_k_model,
    X_train=X_train_bovw_k,
    X_test=X_test_bovw_k,
    y_test=y_test,
    feature_names=bovw_k_feats,  # e.g. ["bovw_0", ..., "prob_k2_0", "prob_k10_0", ...]
    prefix="rq2_bovw_k_xgb"
)

=== SHAP explainability for rq2_bovw_k_xgb ===


 96%|=================== | 1915/2000 [00:14<00:00]       

  - Saved SHAP beeswarm to /content/explainability/rq2_bovw_k_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/explainability/rq2_bovw_k_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/explainability/rq2_bovw_k_xgb_shap_importance.csv
  - Saved local TP waterfall to /content/explainability/rq2_bovw_k_xgb_shap_local_TP_idx1696.png
  - Saved local FP waterfall to /content/explainability/rq2_bovw_k_xgb_shap_local_FP_idx0.png
  - Saved local FN waterfall to /content/explainability/rq2_bovw_k_xgb_shap_local_FN_idx1699.png


In [ ]:
# Load silhouette grid and identify best k by silhouette score
sil_df = pd.read_csv(BOVW_DIR / "silhouette_k_grid.csv")
print("Silhouette grid:\n", sil_df.to_string(index=False))

# Best k = highest silhouette score (k=2 is highest but too coarse for feature diversity;
# use best k >= 10 for meaningful cluster-derived features)
sil_df_filtered = sil_df[sil_df["k"] >= 10]
best_k = int(sil_df_filtered.iloc[sil_df_filtered["silhouette"].idxmax()]["k"])
print(f"\nBest k by silhouette (k>=10): {best_k}")


Silhouette grid:
  k  silhouette  n_sample  random_state  note
 2    0.142945      4000            42   NaN
 5    0.099495      4000            42   NaN
10    0.073887      4000            42   NaN
20    0.042437      4000            42   NaN
30    0.014052      4000            42   NaN
40   -0.000341      4000            42   NaN
50    0.000739      4000            42   NaN

Best k by silhouette (k>=10): 30


In [ ]:
# Raw BoVW only
X_train_bovw = train_df[bovw_cols].values
X_test_bovw  = test_df[bovw_cols].values

bovw_model, bovw_oof, bovw_test_proba, bovw_cv, bovw_test = \
    run_xgb_cv_and_test(X_train_bovw, y_train,
                        X_test_bovw, y_test,
                        model_name="RQ3_BoVW_raw_XGB")

# PCA features only
pca_cols = [c for c in train_df.columns if c.startswith("pca_")]
X_train_pca = train_df[pca_cols].values
X_test_pca  = test_df[pca_cols].values

pca_model, pca_oof, pca_test_proba, pca_cv, pca_test = \
    run_xgb_cv_and_test(X_train_pca, y_train,
                        X_test_pca, y_test,
                        model_name="RQ3_BoVW_PCA_XGB")

print("\nRQ3 BoVW raw TEST metrics:", bovw_test)
print("RQ3 BoVW PCA TEST metrics:", pca_test)


=== RQ3_BoVW_raw_XGB: 5-fold CV (native early stopping) ===
Fold 1: AUROC=0.5593, AP=0.5478, best_iter=265
Fold 2: AUROC=0.5606, AP=0.5522, best_iter=300
Fold 3: AUROC=0.5655, AP=0.5503, best_iter=302
Fold 4: AUROC=0.5787, AP=0.5665, best_iter=327
Fold 5: AUROC=0.5652, AP=0.5517, best_iter=328
=== RQ3_BoVW_PCA_XGB: 5-fold CV (native early stopping) ===
Fold 1: AUROC=0.5628, AP=0.5467, best_iter=277
Fold 2: AUROC=0.5802, AP=0.5692, best_iter=347
Fold 3: AUROC=0.5665, AP=0.5526, best_iter=349
Fold 4: AUROC=0.5716, AP=0.5610, best_iter=409
Fold 5: AUROC=0.5599, AP=0.5393, best_iter=287

RQ3 BoVW raw TEST metrics: {'threshold': 0.5, 'AUROC': 0.5930335945394378, 'AP': 0.5879475203481952, 'Accuracy': 0.5687551695616212, 'Precision': 0.5682891764222633, 'Recall': 0.5721670802315963, 'Specificity': 0.5653432588916459, 'F1': 0.5702215352910871, 'TP': 2767, 'TN': 2734, 'FP': 2102, 'FN': 2069}
RQ3 BoVW PCA TEST metrics: {'threshold': 0.5, 'AUROC': 0.5835430447888423, 'AP': 0.5773076952354522, 'Ac

In [ ]:
# RQ3: BoVW-raw
shap_rq3_raw = explain_xgb_with_shap(
    model=bovw_model,
    X_train=X_train_bovw,
    X_test=X_test_bovw,
    y_test=y_test,
    feature_names=bovw_cols,

    prefix="rq3_bovw_raw_xgb"
)

=== SHAP explainability for rq3_bovw_raw_xgb ===


 97%|=================== | 1945/2000 [00:21<00:00]       

  - Saved SHAP beeswarm to /content/explainability/rq3_bovw_raw_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/explainability/rq3_bovw_raw_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/explainability/rq3_bovw_raw_xgb_shap_importance.csv
  - Saved local TP waterfall to /content/explainability/rq3_bovw_raw_xgb_shap_local_TP_idx1696.png
  - Saved local FP waterfall to /content/explainability/rq3_bovw_raw_xgb_shap_local_FP_idx0.png
  - Saved local FN waterfall to /content/explainability/rq3_bovw_raw_xgb_shap_local_FN_idx1704.png


In [ ]:
# RQ3: BoVW+PCA
shap_rq3_pca = explain_xgb_with_shap(
    model=pca_model,
    X_train=X_train_pca,
    X_test=X_test_pca,
    y_test=y_test,
    feature_names=pca_cols,  # e.g. ["pca_0", ..., "pca_63"]
    prefix="rq3_bovw_pca_xgb"
)

=== SHAP explainability for rq3_bovw_pca_xgb ===


 94%|=================== | 1886/2000 [00:13<00:00]       

  - Saved SHAP beeswarm to /content/explainability/rq3_bovw_pca_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/explainability/rq3_bovw_pca_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/explainability/rq3_bovw_pca_xgb_shap_importance.csv
  - Saved local TP waterfall to /content/explainability/rq3_bovw_pca_xgb_shap_local_TP_idx1697.png
  - Saved local FP waterfall to /content/explainability/rq3_bovw_pca_xgb_shap_local_FP_idx0.png
  - Saved local FN waterfall to /content/explainability/rq3_bovw_pca_xgb_shap_local_FN_idx1696.png


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import glob, os

IMG_SIZE = (256, 256)
BATCH_SIZE = 64

def list_images_with_labels(root_dir):
    paths, labels = [], []
    for label_name, label_val in [("real", 0), ("fake", 1)]:
        for p in glob.glob(os.path.join(root_dir, label_name, "**", "*.*"), recursive=True):
            ext = os.path.splitext(p)[1].lower()
            if ext in [".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"]:
                paths.append(p)
                labels.append(label_val)
    return np.array(paths), np.array(labels)

train_img_paths, train_img_labels = list_images_with_labels(str(DST_AFHQ_TRAIN))
test_img_paths,  test_img_labels  = list_images_with_labels(str(DST_AFHQ_TEST))

print("Train images:", len(train_img_paths), "Test images:", len(test_img_paths))


Train images: 22572 Test images: 9672


In [ ]:
# ===========================
# Image-path ordering alignment (CRITICAL)
# - Sort image path arrays by filename
# - Assert perfect alignment with train_df/test_df filename order
# - Create y_*_img_order arrays for image-derived ordering (do NOT overwrite y_test_df)
# ===========================
from pathlib import Path
import numpy as np

def _sort_paths_and_labels(paths, labels):
    paths = np.array(paths)
    labels = np.array(labels).astype(int)
    order = np.argsort([Path(p).name for p in paths])
    return paths[order], labels[order]

train_img_paths, train_img_labels = _sort_paths_and_labels(train_img_paths, train_img_labels)
test_img_paths,  test_img_labels  = _sort_paths_and_labels(test_img_paths,  test_img_labels)

# Canonical filename lists
train_filenames_df = train_df["filename"].astype(str).tolist()
test_filenames_df  = test_df["filename"].astype(str).tolist()

train_filenames_img = [Path(p).name for p in train_img_paths]
test_filenames_img  = [Path(p).name for p in test_img_paths]

# Hard alignment assertions
assert train_filenames_df == train_filenames_img, "TRAIN ordering mismatch: df vs image paths"
assert test_filenames_df  == test_filenames_img,  "TEST ordering mismatch: df vs image paths"

# Image-order labels (should match df labels after alignment)
y_train_img_order = train_img_labels.astype(int)
y_test_img_order  = test_img_labels.astype(int)

assert np.array_equal(y_train, y_train_img_order), "TRAIN labels mismatch after alignment"
assert np.array_equal(y_test_df, y_test_img_order), "TEST labels mismatch after alignment"

print("✅ Alignment OK: df rows, image paths, and labels match by filename.")


✅ Alignment OK: df rows, image paths, and labels match by filename.


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

IMG_SIZE   = (256, 256)
INPUT_SHAPE = IMG_SIZE + (3,)

def build_small_cnn(input_shape=INPUT_SHAPE):
    # Set global random seeds for reproducibility before building the model.
    # Placing the seed call here ensures it fires on every fold and final-train
    # invocation, eliminating the fold-level non-determinism seen in prior runs.
    tf.random.set_seed(42)
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # --- Optional augmentation (can comment out if you want) ---
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        # -----------------------------------------------------------

        layers.Rescaling(1./255),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.GlobalAveragePooling2D(),

        layers.Dense(64, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc")
        ]
    )
    return model



In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 64

def decode_and_resize(path, label):
    # path is a tf.string
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    # model will rescale to [0,1], so keep as uint8/float32 here
    img = tf.cast(img, tf.float32)
    label = tf.cast(label, tf.float32)
    return img, label

def make_dataset(paths, labels, shuffle=False, batch_size=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(decode_and_resize, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(AUTOTUNE)
    return ds


In [ ]:
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_recall_fscore_support
import time
import numpy as np
import tensorflow as tf

def run_cnn_cv_and_test(train_paths, train_labels,
                        test_paths, test_labels,
                        n_splits=5, epochs=16, batch_size=64,
                        random_state=42,
                        final_val_size=0.1,
                        es_patience=2):
    """
    CNN protocol (FIXED):
      - Outer CV uses EarlyStopping(val_loss) with restore_best_weights
      - Final training ALSO uses the same EarlyStopping protocol on a held-out val split
    Returns:
      final_model, oof_proba, test_proba, cv_summary, test_metrics
    """
    train_paths = np.array(train_paths)
    train_labels = np.array(train_labels).astype(int)
    test_paths = np.array(test_paths)
    test_labels = np.array(test_labels).astype(int)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_proba = np.zeros(len(train_labels), dtype=float)
    fold_stats = []

    print("\n=== CNN: 5-fold CV ===")
    fold_idx = 0

    for tr_idx, val_idx in skf.split(train_paths, train_labels):
        fold_idx += 1
        print(f"\n--- Fold {fold_idx}/{n_splits} ---")

        tr_paths, val_paths = train_paths[tr_idx], train_paths[val_idx]
        tr_labels, val_labels = train_labels[tr_idx], train_labels[val_idx]

        train_ds = make_dataset(tr_paths, tr_labels, shuffle=True,  batch_size=batch_size)
        val_ds   = make_dataset(val_paths, val_labels, shuffle=False, batch_size=batch_size)

        model = build_small_cnn()

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=es_patience, restore_best_weights=True
            )
        ]

        t0 = time.time()
        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=1,
            callbacks=callbacks
        )
        train_time = time.time() - t0

        # Get probabilities on validation set
        val_proba = model.predict(val_ds, verbose=0).ravel()
        oof_proba[val_idx] = val_proba

        val_pred = (val_proba >= 0.5).astype(int)

        fold_auc = roc_auc_score(val_labels, val_proba)
        fold_acc = accuracy_score(val_labels, val_pred)
        fold_prec, fold_rec, fold_f1, _ = precision_recall_fscore_support(
            val_labels, val_pred, average="binary", zero_division=0
        )

        print(f"Fold {fold_idx} AUC={fold_auc:.3f}, F1={fold_f1:.3f}, "
              f"Acc={fold_acc:.3f}, Prec={fold_prec:.3f}, Rec={fold_rec:.3f}, "
              f"train_time={train_time:.2f}s")

        fold_stats.append({
            "fold": fold_idx,
            "auc": fold_auc,
            "accuracy": fold_acc,
            "precision": fold_prec,
            "recall": fold_rec,
            "f1": fold_f1,
            "train_time_s": train_time
        })

    # CV summary
    cv_summary = {}
    for key in ["auc", "accuracy", "precision", "recall", "f1", "train_time_s"]:
        vals = [fs[key] for fs in fold_stats]
        cv_summary[f"{key}_mean"] = float(np.mean(vals))
        cv_summary[f"{key}_std"]  = float(np.std(vals))

    print("\n=== CNN CV summary ===")
    for k, v in cv_summary.items():
        print(f"  {k}: {v:.4f}")

    # === Final train w/ SAME early stopping protocol ===
    # Create a deterministic stratified train/val split from TRAIN only
    sss = StratifiedShuffleSplit(n_splits=1, test_size=final_val_size, random_state=random_state)
    tr_idx, val_idx = next(sss.split(train_paths, train_labels))

    final_tr_paths, final_val_paths = train_paths[tr_idx], train_paths[val_idx]
    final_tr_labels, final_val_labels = train_labels[tr_idx], train_labels[val_idx]

    final_train_ds = make_dataset(final_tr_paths, final_tr_labels, shuffle=True,  batch_size=batch_size)
    final_val_ds   = make_dataset(final_val_paths, final_val_labels, shuffle=False, batch_size=batch_size)
    test_ds        = make_dataset(test_paths,        test_labels,      shuffle=False, batch_size=batch_size)

    final_model = build_small_cnn()
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=es_patience, restore_best_weights=True
        )
    ]

    t0 = time.time()
    final_model.fit(
        final_train_ds,
        validation_data=final_val_ds,
        epochs=epochs,
        verbose=1,
        callbacks=callbacks
    )
    full_train_time = time.time() - t0

    # Inference time
    t0 = time.time()
    test_proba = final_model.predict(test_ds, verbose=0).ravel()
    inference_time = time.time() - t0
    time_per_image = inference_time / max(1, len(test_labels))

    test_pred = (test_proba >= 0.5).astype(int)

    test_auc = roc_auc_score(test_labels, test_proba)
    test_ap  = average_precision_score(test_labels, test_proba)
    test_acc = accuracy_score(test_labels, test_pred)
    test_prec, test_rec, test_f1, _ = precision_recall_fscore_support(
        test_labels, test_pred, average="binary", zero_division=0
    )

    test_metrics = {
        "auc": float(test_auc),
        "ap":  float(test_ap),
        "accuracy": float(test_acc),
        "precision": float(test_prec),
        "recall": float(test_rec),
        "f1": float(test_f1),
        "full_train_time_s": float(full_train_time),
        "inference_time_s_total": float(inference_time),
        "inference_time_s_per_image": float(time_per_image),
        "n_params": int(final_model.count_params()),
        "early_stopping": True,
        "es_patience": int(es_patience),
        "final_val_size": float(final_val_size)
    }

    print("\n=== CNN TEST metrics ===")
    print(f"  AUC={test_auc:.3f}, F1={test_f1:.3f}, Acc={test_acc:.3f}, "
          f"Prec={test_prec:.3f}, Rec={test_rec:.3f}")
    print(f"  Train_time_full={full_train_time:.2f}s, "
          f"Total_infer_time={inference_time:.2f}s, "
          f"Per_image={time_per_image*1000:.2f} ms")
    print(f"  Model parameters: {final_model.count_params()}")
    print(f"  EarlyStopping: val_loss, patience={es_patience}, restore_best_weights=True")

    return final_model, oof_proba, test_proba, cv_summary, test_metrics


In [ ]:
cnn_model, cnn_oof_proba, cnn_test_proba, cnn_cv, cnn_test = \
    run_cnn_cv_and_test(train_img_paths, train_img_labels,
                        test_img_paths,  test_img_labels,
                        n_splits=5, epochs=16, batch_size=64)

print("\nCNN CV summary:", cnn_cv)
print("CNN TEST metrics:", cnn_test)



=== CNN: 5-fold CV ===

--- Fold 1/5 ---
Epoch 1/16
283/283 ━━━━━━━━━━━━━━━━━━━━ 22s 40ms/step - accuracy: 0.5075 - auc: 0.5064 - loss: 0.6933 - val_accuracy: 0.5134 - val_auc: 0.5000 - val_loss: 0.6931
Epoch 2/16
283/283 ━━━━━━━━━━━━━━━━━━━━ 13s 36ms/step - accuracy: 0.4988 - auc: 0.4936 - loss: 0.6933 - val_accuracy: 0.5019 - val_auc: 0.5027 - val_loss: 0.6931
Epoch 3/16
283/283 ━━━━━━━━━━━━━━━━━━━━ 12s 36ms/step - accuracy: 0.5141 - auc: 0.4955 - loss: 0.6933 - val_accuracy: 0.4997 - val_auc: 0.5000 - val_loss: 0.6932
Fold 1 AUC=0.529, F1=0.612, Acc=0.513, Prec=0.509, Rec=0.768, train_time=46.86s

--- Fold 2/5 ---
Epoch 1/16
283/283 ━━━━━━━━━━━━━━━━━━━━ 15s 38ms/step - accuracy: 0.4993 - auc: 0.4921 - loss: 0.6937 - val_accuracy: 0.4997 - val_auc: 0.5000 - val_loss: 0.6931
Epoch 2/16
283/283 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/step - accuracy: 0.5006 - auc: 0.4971 - loss: 0.6932 - val_accuracy: 0.5393 - val_auc: 0.5475 - val_loss: 0.6921
Epoch 3/16
283/283 ━━━━━━━━━━━━━━━━━━━━ 12s 37ms/s

In [ ]:
import numpy as np
import tensorflow as tf

#train_img_paths, train_img_labels = list_images_with_labels(str(DST_AFHQ_TRAIN))
#test_img_paths,  test_img_labels  = list_images_with_labels(str(DST_AFHQ_TEST))

# Load all test images into memory for LIME (ok for ~10k images at 128x128)
def load_image_raw(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img.set_shape((*IMG_SIZE, 3))
    return img.numpy()  # float32, 0-255

X_test_imgs = np.stack([load_image_raw(p) for p in test_img_paths])
y_test_img_order = np.array(test_img_labels, dtype=int)
print("X_test_imgs:", X_test_imgs.shape, "y_test_img_order:", y_test_img_order.shape)


X_test_imgs: (9672, 256, 256, 3) y_test_img_order: (9672,)


In [ ]:
def cnn_classifier_fn(images: np.ndarray):
    """
    images: (N, H, W, 3) in 0-255 (uint8 or float).
    Returns probs for shape (N, 2): [p(real), p(fake)].
    """
    imgs = tf.convert_to_tensor(images, dtype=tf.float32)
    preds = cnn_model.predict(imgs, verbose=0).reshape(-1)  # sigmoid -> p(fake)
    preds_fake = preds
    preds_real = 1.0 - preds_fake
    return np.vstack([preds_real, preds_fake]).T


In [ ]:
from sklearn.metrics import roc_auc_score

# Use the image-order label array explicitly to avoid any accidental overwrite
print("CNN Test AUC:", roc_auc_score(y_test_img_order, cnn_test_proba))

# Thresholded predictions (required for TP/FP selection)
cnn_test_pred = (cnn_test_proba >= 0.5).astype(int)

tp_idx = np.where((y_test_img_order == 1) & (cnn_test_pred == 1))[0]
tn_idx = np.where((y_test_img_order == 0) & (cnn_test_pred == 0))[0]
fp_idx = np.where((y_test_img_order == 0) & (cnn_test_pred == 1))[0]
fn_idx = np.where((y_test_img_order == 1) & (cnn_test_pred == 0))[0]


CNN Test AUC: 0.7504717812915404


In [ ]:
from lime import lime_image
from skimage.segmentation import mark_boundaries
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
import numpy as np
import pandas as pd

lime_img_explainer = lime_image.LimeImageExplainer()

EXPLAIN_IMG_DIR = Path("/content/stargan-v2/explainability/lime_cnn_images")
EXPLAIN_IMG_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)

def pick_some(idxs, k=2):
    idxs = np.array(idxs)
    if len(idxs) == 0:
        return []
    if len(idxs) <= k:
        return idxs.tolist()
    return rng.choice(idxs, size=k, replace=False).tolist()

# Build a metadata lookup keyed by filename so LIME captions match the explained image
test_meta = test_df.set_index("filename", drop=False)

def explain_cnn_image(idx):
    # image + meta
    path = str(test_img_paths[idx])
    fname = Path(path).name

    img = X_test_imgs[idx].astype(np.uint8)
    true_label = int(y_test_img_order[idx])
    pred_prob = float(cnn_test_proba[idx])
    pred_label = int(pred_prob >= 0.5)

    if fname in test_meta.index:
        row = test_meta.loc[fname]
        fake_tech = row["fake_technique"] if "fake_technique" in test_meta.columns else "unknown"
        label_name = row["label_name"] if "label_name" in test_meta.columns else ("fake" if true_label==1 else "real")
    else:
        fake_tech = "unknown"
        label_name = ("fake" if true_label==1 else "real")

    print("="*80)
    print(f"CNN LIME for filename: {fname}")
    print(f"path           : {path}")
    print(f"true label     : {true_label} ({label_name})")
    print(f"pred label     : {pred_label} ({'fake' if pred_label==1 else 'real'})")
    print(f"pred prob(fake): {pred_prob:.3f}")
    if true_label == 1:
        print(f"fake_technique : {fake_tech}")
    print("-"*80)

    explanation = lime_img_explainer.explain_instance(
        image=img,
        classifier_fn=cnn_classifier_fn,
        top_labels=2,
        hide_color=0,
        num_samples=1000
    )

    # --- FIX: Build a visible coloured overlay using superpixel weights ---
    # Get the segment map and per-segment weights for the "fake" class (label=1)
    segments = explanation.segments  # (H, W) integer array of segment IDs
    local_exp = explanation.local_exp.get(1, [])  # list of (segment_id, weight) for fake class

    # Build weight map: positive = green (supports fake), negative = red (supports real)
    weight_map = np.zeros(segments.shape, dtype=np.float32)
    if local_exp:
        weights = np.array([w for _, w in local_exp])
        max_abs = np.abs(weights).max() + 1e-9
        for seg_id, w in local_exp:
            weight_map[segments == seg_id] = w / max_abs  # normalise to [-1, 1]

    # Build RGBA overlay: green for positive, red for negative, alpha proportional to |weight|
    overlay = np.zeros(img.shape[:2] + (4,), dtype=np.float32)
    overlay[weight_map > 0] = [0, 1, 0, 0]   # green channel
    overlay[weight_map < 0] = [1, 0, 0, 0]   # red channel
    overlay[..., 3] = np.clip(np.abs(weight_map) * 0.55, 0, 0.55)  # semi-transparent

    # Panel: original | LIME overlay with boundaries
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    axes[0].imshow(img)
    axes[0].set_title(f"Original\ntrue={label_name}, pred={'fake' if pred_label==1 else 'real'} ({pred_prob:.2f})",
                      fontsize=9)
    axes[0].axis("off")

    img_float = img.astype(np.float32) / 255.0
    # Draw superpixel boundaries on the base image
    bounded = mark_boundaries(img_float, segments, color=(1, 1, 0), mode="thick")
    axes[1].imshow(bounded)
    axes[1].imshow(overlay)   # coloured weight overlay on top
    axes[1].set_title("LIME — green: supports 'fake', red: supports 'real'", fontsize=9)
    axes[1].axis("off")

    # Add a simple legend
    from matplotlib.patches import Patch
    legend_elems = [Patch(facecolor="green", alpha=0.55, label="Supports fake"),
                    Patch(facecolor="red",   alpha=0.55, label="Supports real")]
    axes[1].legend(handles=legend_elems, loc="lower right", fontsize=7)

    out_name = EXPLAIN_IMG_DIR / f"lime_cnn_{fname}_true{true_label}_pred{pred_label}.png"
    plt.tight_layout()
    plt.savefig(out_name, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved LIME image explanation to {out_name}\n")


In [ ]:
import numpy as np

# Flatten proba if needed
scores = cnn_test_proba.ravel()

# Indices of real / fake
idx_real = np.where(y_test_img_order == 0)[0]
idx_fake = np.where(y_test_img_order == 1)[0]

def top_k(idxs, score_vec, k, reverse=False):
    """Return up to k indices from idxs ordered by score."""
    idxs = np.array(idxs)
    if len(idxs) == 0:
        return []
    order = np.argsort(score_vec[idxs])
    if reverse:  # highest scores first
        order = order[::-1]
    sel = idxs[order[:k]]
    return sel.tolist()

# "Most confident" groups, even if model is bad
TP_like = top_k(idx_fake, scores, k=5, reverse=True)   # true fake, highest P(fake)
FN_like = top_k(idx_fake, scores, k=5, reverse=False)  # true fake, lowest P(fake)
TN_like = top_k(idx_real, scores, k=5, reverse=False)  # true real, lowest P(fake)
FP_like = top_k(idx_real, scores, k=5, reverse=True)   # true real, highest P(fake)

example_groups = {
    "TP_like_fake":        TP_like,
    "TN_like_real":        TN_like,
    "FP_real_as_fake":     FP_like,
    "FN_fake_as_real":     FN_like,
}

for group_name, idxs in example_groups.items():
    print(f"\n### {group_name} examples")
    if not idxs:
        print("  (none found)")
        continue
    for idx in idxs:
        explain_cnn_image(idx)



### TP_like_fake examples
CNN LIME for filename: swap_like__1f2bed35__pixabay_dog_001144_swap_fdbc7f47__pixabay_dog_001104.jpg
path           : /content/stargan-v2/combined_balanced/test/fake/swap_like/swap_like__1f2bed35__pixabay_dog_001144_swap_fdbc7f47__pixabay_dog_001104.jpg
true label     : 1 (fake)
pred label     : 1 (fake)
pred prob(fake): 1.000
fake_technique : swap_like
--------------------------------------------------------------------------------


  0%|          | 0/1000 [00:00<?, ?it/s]

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_swap_like__1f2bed35__pixabay_dog_001144_swap_fdbc7f47__pixabay_dog_001104.jpg_true1_pred1.png

CNN LIME for filename: swap_like__b4b5199f__pixabay_dog_000619_swap_d36c89b3__pixabay_dog_003557.jpg
path           : /content/stargan-v2/combined_balanced/test/fake/swap_like/swap_like__b4b5199f__pixabay_dog_000619_swap_d36c89b3__pixabay_dog_003557.jpg
true label     : 1 (fake)
pred label     : 1 (fake)
pred prob(fake): 1.000
fake_technique : swap_like
--------------------------------------------------------------------------------


  0%|          | 0/1000 [00:00<?, ?it/s]

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_swap_like__b4b5199f__pixabay_dog_000619_swap_d36c89b3__pixabay_dog_003557.jpg_true1_pred1.png

CNN LIME for filename: splicing__fea4d402__pixabay_cat_003180_splice_c5dd0e75__pixabay_dog_000760.jpg
path           : /content/stargan-v2/combined_balanced/test/fake/splicing/splicing__fea4d402__pixabay_cat_003180_splice_c5dd0e75__pixabay_dog_000760.jpg
true label     : 1 (fake)
pred label     : 1 (fake)
pred prob(fake): 1.000
fake_technique : splicing
--------------------------------------------------------------------------------


...

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_inpaint__bc76a743__pixabay_cat_002962_inpaint.jpg_true1_pred0.png

CNN LIME for filename: splicing__ae139d4c__pixabay_cat_000324_splice_de138245__pixabay_dog_002786.jpg
path           : /content/stargan-v2/combined_balanced/test/fake/splicing/splicing__ae139d4c__pixabay_cat_000324_splice_de138245__pixabay_dog_002786.jpg
true label     : 1 (fake)
pred label     : 0 (real)
pred prob(fake): 0.027
fake_technique : splicing
--------------------------------------------------------------------------------


  0%|          | 0/1000 [00:00<?, ?it/s]

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_splicing__ae139d4c__pixabay_cat_000324_splice_de138245__pixabay_dog_002786.jpg_true1_pred0.png

CNN LIME for filename: splicing__289409d2__flickr_cat_000369_splice_34632125__flickr_wild_001198.jpg
path           : /content/stargan-v2/combined_balanced/test/fake/splicing/splicing__289409d2__flickr_cat_000369_splice_34632125__flickr_wild_001198.jpg
true label     : 1 (fake)
pred label     : 0 (real)
pred prob(fake): 0.037
fake_technique : splicing
--------------------------------------------------------------------------------


  0%|          | 0/1000 [00:00<?, ?it/s]

Saved LIME image explanation to /content/stargan-v2/explainability/lime_cnn_images/lime_cnn_splicing__289409d2__flickr_cat_000369_splice_34632125__flickr_wild_001198.jpg_true1_pred0.png



In [ ]:
# ===========================
# RQ5 Ensemble (CRITICAL FIX)
# - Convert every proba array into a DF keyed by filename
# - Merge on filename before stacking / fitting meta-learner
# ===========================
from pathlib import Path
import pandas as pd
import numpy as np

# Tabular model predictions are already in train_df/test_df order (sorted by filename).
train_pred_tab = pd.DataFrame({
    "filename": train_df["filename"].astype(str),
    "forensic_proba": forensic_oof,
    "bovw_k_proba":   bovw_k_oof,
    "bovw_raw_proba": bovw_oof,
    "pca_proba":      pca_oof,
})

test_pred_tab = pd.DataFrame({
    "filename": test_df["filename"].astype(str),
    "forensic_proba": forensic_test_proba,
    "bovw_k_proba":   bovw_k_test_proba,
    "bovw_raw_proba": bovw_test_proba,
    "pca_proba":      pca_test_proba,
})

# CNN predictions keyed by image filenames (after alignment, these should match df filenames)
train_pred_cnn = pd.DataFrame({
    "filename": [Path(p).name for p in train_img_paths],
    "cnn_proba": cnn_oof_proba
})
test_pred_cnn = pd.DataFrame({
    "filename": [Path(p).name for p in test_img_paths],
    "cnn_proba": cnn_test_proba
})

# Merge predictions
ensemble_train_df = train_pred_tab.merge(train_pred_cnn, on="filename", how="inner")
ensemble_test_df  = test_pred_tab.merge(test_pred_cnn,  on="filename", how="inner")

# Alignment assertions
assert set(ensemble_train_df["filename"]) == set(train_df["filename"]), "TRAIN ensemble merge lost rows"
assert set(ensemble_test_df["filename"])  == set(test_df["filename"]),  "TEST ensemble merge lost rows"

ensemble_train_df = ensemble_train_df.sort_values("filename").reset_index(drop=True)
ensemble_test_df  = ensemble_test_df.sort_values("filename").reset_index(drop=True)

assert ensemble_train_df["filename"].tolist() == train_df["filename"].tolist(), "TRAIN ensemble order mismatch after merge"
assert ensemble_test_df["filename"].tolist()  == test_df["filename"].tolist(),  "TEST ensemble order mismatch after merge"

# Final feature matrices (meta-learner inputs)
ensemble_feature_cols = ["forensic_proba", "bovw_k_proba", "bovw_raw_proba", "pca_proba", "cnn_proba"]

ensemble_X_train_df = ensemble_train_df[ensemble_feature_cols].copy()
ensemble_X_test_df  = ensemble_test_df[ensemble_feature_cols].copy()

assert not ensemble_X_train_df.isna().any().any(), "NaNs in ensemble_X_train_df"
assert not ensemble_X_test_df.isna().any().any(),  "NaNs in ensemble_X_test_df"

ens_model, ens_oof, ens_test_proba, ens_cv, ens_test =     run_xgb_cv_and_test(ensemble_X_train_df.values, y_train,
                        ensemble_X_test_df.values,  y_test_df,
                        model_name="RQ5_Ensemble_XGB")

# Stable aliases for downstream reporting / dissertation tables
ensemble_model = ens_model
ensemble_oof = ens_oof
ensemble_test_proba = ens_test_proba
ensemble_cv = ens_cv
ensemble_test = ens_test

print("\nRQ5 Ensemble CV summary:", ensemble_cv)
print("RQ5 Ensemble TEST metrics:", ensemble_test)


=== RQ5_Ensemble_XGB: 5-fold CV (native early stopping) ===
Fold 1: AUROC=0.6511, AP=0.6828, best_iter=709
Fold 2: AUROC=0.7690, AP=0.7997, best_iter=554
Fold 3: AUROC=0.7679, AP=0.7929, best_iter=601
Fold 4: AUROC=0.7840, AP=0.8116, best_iter=660
Fold 5: AUROC=0.7741, AP=0.7938, best_iter=549

RQ5 Ensemble CV summary: {'model': 'RQ5_Ensemble_XGB', 'n_splits': 5, 'AUROC_mean': 0.7492317714390491, 'AUROC_std': 0.05520731647778704, 'AP_mean': 0.7761437383239066, 'AP_std': 0.05270413618398842}
RQ5 Ensemble TEST metrics: {'threshold': 0.5, 'AUROC': 0.7773663080384845, 'AP': 0.8025583707010949, 'Accuracy': 0.7149503722084367, 'Precision': 0.7723342939481268, 'Recall': 0.6095947063688999, 'Specificity': 0.8203060380479735, 'F1': 0.6813821795908933, 'TP': 2948, 'TN': 3967, 'FP': 869, 'FN': 1888}


In [ ]:
# Example: RQ5 ensemble model
# Assume:
#   xgb_forensic_full : fitted XGB model
#   X_forensic_train, X_forensic_test
#   y_train, y_test
#   forensic_cols: list of forensic feature column names

shap_rq1 = explain_xgb_with_shap(
    model=ensemble_model,
    X_train=ensemble_X_train_df,
    X_test=ensemble_X_test_df,
    y_test=y_test,
    feature_names=ensemble_feature_cols,
    prefix="rq5_ensemble_xgb"
)

=== SHAP explainability for rq5_ensemble_xgb ===


100%|===================| 1992/2000 [00:36<00:00]       

  - Saved SHAP beeswarm to /content/explainability/rq5_ensemble_xgb_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/explainability/rq5_ensemble_xgb_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/explainability/rq5_ensemble_xgb_shap_importance.csv
  - Saved local TP waterfall to /content/explainability/rq5_ensemble_xgb_shap_local_TP_idx1697.png
  - Saved local FP waterfall to /content/explainability/rq5_ensemble_xgb_shap_local_FP_idx9.png
  - Saved local FN waterfall to /content/explainability/rq5_ensemble_xgb_shap_local_FN_idx1696.png


In [ ]:
import numpy as np
import pandas as pd

# Where to save lift tables
LIFT_OUT_DIR = DRIVE / "output" / "lift_tables"
LIFT_OUT_DIR.mkdir(parents=True, exist_ok=True)

def compute_lift_table(
    y_true,
    y_score,
    base_df,
    model_name: str,
    n_bins: int = 10,
    out_dir: Path = LIFT_OUT_DIR,
):
    """
    Build a decile-based lift table on the TEST set.

    Interpreting lift:
      - Lift is a ranking-quality diagnostic: "how many true fakes are captured if we review the top X% highest-scored items?"
      - If a technique is absent from the top decile, interpret this as reduced detectability under the current construction,
        not as an intentional deprioritization of that technique.

    IMPORTANT: Only compute after filename/label alignment checks pass for every model output.

    y_true  : array-like of 0/1 labels (1 = deepfake)
    y_score : array-like of predicted probabilities for class 1 (deepfake)
    base_df : DataFrame with at least ['filename','label','label_name','fake_technique']

    """
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)

    if y_true.shape[0] != y_score.shape[0]:
        raise ValueError("y_true and y_score must have the same length")

    df = base_df.copy().reset_index(drop=True)
    if len(df) != len(y_true):
        raise ValueError("base_df length must match y_true/y_score length")

    df["y_true"] = y_true
    df["score"] = y_score

    # Sort by descending predicted probability (highest-risk first)
    df = df.sort_values("score", ascending=False).reset_index(drop=True)
    N = len(df)
    df["rank"] = np.arange(1, N + 1)

    # Assign deciles: 1 = top 10% highest score, 10 = lowest 10%
    df["decile"] = ((df["rank"] - 1) / (N / n_bins)).astype(int) + 1
    df.loc[df["decile"] > n_bins, "decile"] = n_bins

    # Overall target rate (deepfake rate) for lift calculations
    base_rate = df["y_true"].mean()
    total_pos = df["y_true"].sum()

    # Basic per-decile stats
    grouped = df.groupby("decile", as_index=False)
    lift_df = grouped["y_true"].agg(
        n="size",
        n_fake="sum",
    )
    lift_df["n_real"] = lift_df["n"] - lift_df["n_fake"]
    lift_df["pct_fake"] = lift_df["n_fake"] / lift_df["n"].replace(0, np.nan)
    lift_df["pct_real"] = lift_df["n_real"] / lift_df["n"].replace(0, np.nan)

    # Cumulative stats and lift
    lift_df["cum_n"] = lift_df["n"].cumsum()
    lift_df["cum_fake"] = lift_df["n_fake"].cumsum()
    lift_df["cum_fake_rate"] = lift_df["cum_fake"] / (total_pos if total_pos > 0 else np.nan)
    lift_df["pop_frac"] = lift_df["cum_n"] / float(N)

    # Per-decile lift vs overall deepfake rate
    lift_df["lift"] = lift_df["pct_fake"] / (base_rate if base_rate > 0 else np.nan)
    # Cumulative lift (how much better than random by this decile)
    lift_df["cum_lift"] = lift_df["cum_fake_rate"] / lift_df["pop_frac"].replace(0, np.nan)

    # Per-deepfake-method breakdown (counts and % among fakes in that decile)
    if "fake_technique" in df.columns:
        # Only look at fakes for technique stats
        fake_only = df[df["y_true"] == 1].copy()
        if not fake_only.empty:
            tech_tab = (
                fake_only
                .pivot_table(
                    index="decile",
                    columns="fake_technique",
                    values="y_true",
                    aggfunc="size",
                    fill_value=0,
                )
            )
            # Counts per technique
            tech_tab_counts = tech_tab.add_prefix("n_fake_")
            tech_tab_counts.reset_index(inplace=True)

            # Merge counts into lift_df
            lift_df = lift_df.merge(tech_tab_counts, on="decile", how="left")
            lift_df.fillna(0, inplace=True)

            # Percent-of-fakes per decile for each technique
            fake_cols = [c for c in lift_df.columns if c.startswith("n_fake_")]
            for c in fake_cols:
                pct_col = c.replace("n_fake_", "pct_fake_")
                lift_df[pct_col] = lift_df[c] / lift_df["n_fake"].replace(0, np.nan)

    # Save to CSV
    out_path = out_dir / f"lift_{model_name}_test.csv"
    lift_df.to_csv(out_path, index=False)
    print(f"[LIFT] Saved {model_name} lift table to: {out_path}")

    return lift_df


In [ ]:
# === Build lift tables on TEST for each model ===

# Sanity check that all proba arrays line up with test_df
print("Test length:", len(test_df))
print("  y_test_df:", len(y_test_df))
print("  forensic_test_proba:", len(forensic_test_proba))
print("  bovw_k_test_proba:", len(bovw_k_test_proba))
print("  bovw_test_proba:", len(bovw_test_proba))
print("  pca_test_proba:", len(pca_test_proba))
print("  cnn_test_proba:", len(cnn_test_proba))
print("  ens_test_proba:", len(ens_test_proba))

lift_rq1_forensic = compute_lift_table(
    y_true=y_test_df,
    y_score=forensic_test_proba,
    base_df=test_df,
    model_name="RQ1_Forensic_XGB",
)

lift_rq2_bovw_k = compute_lift_table(
    y_true=y_test_df,
    y_score=bovw_k_test_proba,
    base_df=test_df,
    model_name="RQ2_BoVW_plus_Kmeans_XGB",
)

lift_rq3_bovw_raw = compute_lift_table(
    y_true=y_test_df,
    y_score=bovw_test_proba,
    base_df=test_df,
    model_name="RQ3_BoVW_raw_XGB",
)

lift_rq3_bovw_pca = compute_lift_table(
    y_true=y_test_df,
    y_score=pca_test_proba,
    base_df=test_df,
    model_name="RQ3_BoVW_PCA_XGB",
)

lift_rq4_cnn = compute_lift_table(
    y_true=y_test_df,
    y_score=cnn_test_proba,
    base_df=test_df,
    model_name="RQ4_CNN",
)

lift_rq5_ensemble = compute_lift_table(
    y_true=y_test_df,
    y_score=ens_test_proba,
    base_df=test_df,
    model_name="RQ5_Ensemble_XGB",
)

# Quick check
display(lift_rq5_ensemble)


Test length: 9672
  y_test_df: 9672
  forensic_test_proba: 9672
  bovw_k_test_proba: 9672
  bovw_test_proba: 9672
  pca_test_proba: 9672
  cnn_test_proba: 9672
  ens_test_proba: 9672
[LIFT] Saved RQ1_Forensic_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ1_Forensic_XGB_test.csv
[LIFT] Saved RQ2_BoVW_plus_Kmeans_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ2_BoVW_plus_Kmeans_XGB_test.csv
[LIFT] Saved RQ3_BoVW_raw_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ3_BoVW_raw_XGB_test.csv
[LIFT] Saved RQ3_BoVW_PCA_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ3_BoVW_PCA_XGB_test.csv
[LIFT] Saved RQ4_CNN lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ4_CNN_test.csv
[LIFT] Saved RQ5_Ensemble_XGB lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ5_Ensemble_XGB_test.csv


,decile,n,n_fake,n_real,pct_fake,pct_real,cum_n,cum_fake,cum_fake_rate,pop_frac,...,n_fake_postproc,n_fake_splicing,n_fake_stargan_tiles,n_fake_swap_like,pct_fake_copy_move,pct_fake_inpaint,pct_fake_postproc,pct_fake_splicing,pct_fake_stargan_tiles,pct_fake_swap_like
0,1,968,928,40,0.958678,0.041322,968,928,0.191894,0.100083,...,340,219,33,220,0.121767,0.003233,0.366379,0.235991,0.035560,0.237069
1,2,967,796,171,0.823164,0.176836,1935,1724,0.356493,0.200062,...,70,145,224,143,0.224874,0.043970,0.087940,0.182161,0.281407,0.179648
2,3,967,694,273,0.717684,0.282316,2902,2418,0.500000,0.300041,...,67,113,206,118,0.200288,0.073487,0.096542,0.162824,0.296830,0.170029
3,4,967,555,412,0.573940,0.426060,3869,2973,0.614764,0.400021,...,74,76,181,77,0.169369,0.095495,0.133333,0.136937,0.326126,0.138739
4,5,967,434,533,0.448811,0.551189,4836,3407,0.704508,0.500000,...,72,67,58,59,0.198157,0.211982,0.165899,0.154378,0.133641,0.135945
5,6,968,383,585,0.395661,0.604339,5804,3790,0.783706,0.600083,...,71,56,44,50,0.172324,0.250653,0.185379,0.146214,0.114883,0.130548
6,7,967,348,619,0.359876,0.640124,6771,4138,0.855666,0.700062,...,56,54,25,49,0.129310,0.341954,0.160920,0.155172,0.071839,0.140805
7,8,967,261,706,0.269907,0.730093,7738,4399,0.909636,0.800041,...,22,26,22,28,0.153257,0.471264,0.084291,0.099617,0.084291,0.107280
8,9,967,235,732,0.243020,0.756980,8705,4634,0.958230,0.900021,...,23,27,9,27,0.127660,0.506383,0.097872,0.114894,0.038298,0.114894
9,10,967,202,765,0.208893,0.791107,9672,4836,1.000000,1.000000,...,11,23,4,35,0.069307,0.569307,0.054455,0.113861,0.019802,0.173267


In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score

# ----------------------------
# Helpers
# ----------------------------
def _to_numpy(x):
    x = np.asarray(x)
    return x.reshape(-1)

def _validate_binary_y(y):
    y = _to_numpy(y).astype(int)
    uniq = np.unique(y)
    if not set(uniq).issubset({0, 1}):
        raise ValueError(f"y must be binary 0/1. Found values: {uniq}")
    return y

# ----------------------------
# McNemar (exact, binomial)
# ----------------------------
def mcnemar_exact(y_true, p_a, p_b, threshold=0.50):
    """
    Exact McNemar test using binomial test on discordant pairs.
    Compares two classifiers A and B on the SAME examples.
    """
    y_true = _validate_binary_y(y_true)
    p_a = _to_numpy(p_a)
    p_b = _to_numpy(p_b)

    pred_a = (p_a >= threshold).astype(int)
    pred_b = (p_b >= threshold).astype(int)

    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)

    # discordant counts
    n01 = int(np.sum((~correct_a) & ( correct_b)))  # A wrong, B right
    n10 = int(np.sum(( correct_a) & (~correct_b)))  # A right, B wrong
    n = n01 + n10

    if n == 0:
        return {
            "n01_Awrong_Bright": n01,
            "n10_Aright_Bwrong": n10,
            "p_value": 1.0,
            "note": "No discordant pairs; models tie on every example."
        }

    # exact two-sided p-value via binomial test with p=0.5
    try:
        from scipy.stats import binomtest
        pval = binomtest(k=min(n01, n10), n=n, p=0.5, alternative="two-sided").pvalue
    except Exception:
        # fallback: normal approx w/ continuity correction
        import math
        diff = abs(n01 - n10) - 1.0
        chi2 = (diff * diff) / n
        # p-value for chi-square(1)
        # approx using survival function of normal: p ~ 2*(1-Phi(sqrt(chi2)))
        from math import sqrt
        z = sqrt(chi2)
        # crude normal sf
        pval = 2.0 * (1.0 - 0.5 * (1.0 + math.erf(z / np.sqrt(2))))

    return {
        "n01_Awrong_Bright": n01,
        "n10_Aright_Bwrong": n10,
        "p_value": float(pval),
        "threshold": float(threshold),
        "discordant_pairs": int(n)
    }

# ----------------------------
# DeLong test for correlated AUCs (pure numpy)
# Based on the fast DeLong algorithm (Sun & Xu) commonly used in practice.
# ----------------------------
def _compute_midrank(x):
    """Computes midranks of a 1D array."""
    x = _to_numpy(x)
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        # midrank for ties
        mid = 0.5 * (i + j - 1) + 1
        T[i:j] = mid
        i = j
    out = np.empty(N, dtype=float)
    out[J] = T
    return out

def _fast_delong(predictions_sorted_transposed, label_1_count):
    """
    Fast DeLong for 2+ classifiers.
    predictions_sorted_transposed: shape (k_classifiers, n_examples), sorted by ground-truth with positives first
    label_1_count: number of positive examples
    Returns:
      aucs: shape (k_classifiers,)
      delong_cov: covariance matrix of AUC estimates, shape (k, k)
    """
    m = int(label_1_count)
    n = predictions_sorted_transposed.shape[1] - m
    k = predictions_sorted_transposed.shape[0]

    # Compute midranks for positives and negatives per classifier
    tx = np.zeros((k, m))
    ty = np.zeros((k, n))
    tz = np.zeros((k, m + n))

    for r in range(k):
        preds = predictions_sorted_transposed[r, :]
        tz[r, :] = _compute_midrank(preds)
        tx[r, :] = _compute_midrank(preds[:m])
        ty[r, :] = _compute_midrank(preds[m:])

    aucs = (tz[:, :m].sum(axis=1) / m - (m + 1) / 2.0) / n

    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m

    sx = np.cov(v01)
    sy = np.cov(v10)
    delong_cov = sx / m + sy / n

    return aucs, delong_cov

def delong_auc_pvalue(y_true, p_a, p_b):
    """
    Returns AUCs for A and B and DeLong p-value (two-sided) for difference in AUC.
    """
    y_true = _validate_binary_y(y_true)
    p_a = _to_numpy(p_a)
    p_b = _to_numpy(p_b)

    # sort with positives first
    order = np.argsort(-y_true)  # y=1 first
    y_sorted = y_true[order]
    preds = np.vstack([p_a[order], p_b[order]])

    m = int(np.sum(y_sorted))
    if m == 0 or m == len(y_sorted):
        raise ValueError("DeLong test requires both positive and negative samples in y_true.")

    aucs, cov = _fast_delong(preds, m)
    auc_a, auc_b = float(aucs[0]), float(aucs[1])

    # variance of difference
    diff = auc_b - auc_a
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    if var <= 0:
        # numerical edge case
        pval = 1.0
        z = 0.0
    else:
        z = diff / np.sqrt(var)
        try:
            from scipy.stats import norm
            pval = 2 * norm.sf(abs(z))
        except Exception:
            # normal sf fallback
            import math
            pval = 2.0 * (1.0 - 0.5 * (1.0 + math.erf(abs(z) / np.sqrt(2))))

    return {
        "auc_a": auc_a,
        "auc_b": auc_b,
        "auc_diff_b_minus_a": float(diff),
        "z": float(z),
        "p_value": float(pval)
    }

# ----------------------------
# Bootstrap CI for delta metrics (AUC and F1 by default)
# ----------------------------
def bootstrap_delta_metrics(y_true, p_a, p_b, n_boot=5000, seed=42, threshold=0.50):
    """
    Bootstraps the difference (B - A) in AUC and F1 with percentile 95% CI.
    Uses paired resampling of rows.
    """
    rng = np.random.default_rng(seed)
    y_true = _validate_binary_y(y_true)
    p_a = _to_numpy(p_a)
    p_b = _to_numpy(p_b)

    n = len(y_true)
    idx = np.arange(n)

    deltas_auc = []
    deltas_f1  = []

    for _ in range(n_boot):
        s = rng.choice(idx, size=n, replace=True)
        y_s = y_true[s]
        # AUC needs both classes present in the bootstrap sample
        if len(np.unique(y_s)) < 2:
            continue

        pa_s = p_a[s]
        pb_s = p_b[s]

        auc_a = roc_auc_score(y_s, pa_s)
        auc_b = roc_auc_score(y_s, pb_s)
        deltas_auc.append(auc_b - auc_a)

        f1_a = f1_score(y_s, (pa_s >= threshold).astype(int))
        f1_b = f1_score(y_s, (pb_s >= threshold).astype(int))
        deltas_f1.append(f1_b - f1_a)

    deltas_auc = np.array(deltas_auc, dtype=float)
    deltas_f1  = np.array(deltas_f1, dtype=float)

    def ci(x):
        return (float(np.quantile(x, 0.025)), float(np.quantile(x, 0.975)), float(np.mean(x)))

    auc_ci = ci(deltas_auc)
    f1_ci  = ci(deltas_f1)

    return {
        "n_boot_requested": int(n_boot),
        "n_boot_used_auc_f1": int(len(deltas_auc)),
        "threshold": float(threshold),
        "delta_auc_mean": auc_ci[2],
        "delta_auc_ci95": (auc_ci[0], auc_ci[1]),
        "delta_f1_mean": f1_ci[2],
        "delta_f1_ci95": (f1_ci[0], f1_ci[1]),
    }

# ============================
# RUN THE TESTS (RQ5 vs RQ1)
# ============================

# Your variables:
# Use df-derived holdout labels for all paired statistical tests
y = y_test_df
p_rq1 = forensic_test_proba
p_rq5 = ens_test_proba

print("AUC RQ1 (forensic):", roc_auc_score(y, p_rq1))
print("AUC RQ5 (ensemble):", roc_auc_score(y, p_rq5))
print("F1  RQ1 (thr=0.50):", f1_score(y, (np.asarray(p_rq1) >= 0.50).astype(int)))
print("F1  RQ5 (thr=0.50):", f1_score(y, (np.asarray(p_rq5) >= 0.50).astype(int)))

# 1) DeLong test (AUC diff)
delong_res = delong_auc_pvalue(y, p_rq1, p_rq5)

# 2) McNemar exact test (paired errors) at threshold 0.50
mcnemar_res = mcnemar_exact(y, p_rq1, p_rq5, threshold=0.50)

# 3) Bootstrap CI for delta AUC and delta F1
boot_res = bootstrap_delta_metrics(y, p_rq1, p_rq5, n_boot=5000, seed=42, threshold=0.50)

print("\n=== Statistical Comparison: RQ5 vs RQ1 (holdout test set) ===")
print("DeLong AUC test:", delong_res)
print("McNemar test:", mcnemar_res)
print("Bootstrap delta metrics:", boot_res)

alpha = 0.05
print("\nDecision @ alpha=0.05:")
print("  DeLong significant? ", delong_res["p_value"] < alpha)
print("  McNemar significant?", mcnemar_res["p_value"] < alpha)
print("  Bootstrap AUC CI excludes 0? ", not (boot_res["delta_auc_ci95"][0] <= 0 <= boot_res["delta_auc_ci95"][1]))
print("  Bootstrap F1  CI excludes 0? ", not (boot_res["delta_f1_ci95"][0]  <= 0 <= boot_res["delta_f1_ci95"][1]))


AUC RQ1 (forensic): 0.6386483909621866
AUC RQ5 (ensemble): 0.7773663080384845
F1  RQ1 (thr=0.50): 0.52298991885911
F1  RQ5 (thr=0.50): 0.6813821795908933

=== Statistical Comparison: RQ5 vs RQ1 (holdout test set) ===
DeLong AUC test: {'auc_a': 0.6386483909621866, 'auc_b': 0.7773663080384845, 'auc_diff_b_minus_a': 0.13871791707629788, 'z': 25.140548298801402, 'p_value': 1.7930072318289276e-139}
McNemar test: {'n01_Awrong_Bright': 1865, 'n10_Aright_Bwrong': 742, 'p_value': 1.57742317433495e-110, 'threshold': 0.5, 'discordant_pairs': 2607}
Bootstrap delta metrics: {'n_boot_requested': 5000, 'n_boot_used_auc_f1': 5000, 'threshold': 0.5, 'delta_auc_mean': 0.13864384798193788, 'delta_auc_ci95': (0.12787492210781642, 0.14943776278421608), 'delta_f1_mean': 0.15822004531373354, 'delta_f1_ci95': (0.1447800603050231, 0.1717032268867381)}

Decision @ alpha=0.05:
  DeLong significant?  True
  McNemar significant? True
  Bootstrap AUC CI excludes 0?  True
  Bootstrap F1  CI excludes 0?  True


In [ ]:
# ============================
# RUN THE TESTS (RQ4 vs RQ5)
# ============================

# Your variables:
# Use df-derived holdout labels for all paired statistical tests
y = y_test_df
p_rq4 = cnn_test_proba
p_rq5 = ens_test_proba

print("AUC RQ4 (cnn):", roc_auc_score(y, p_rq4))
print("AUC RQ5 (ensemble):", roc_auc_score(y, p_rq5))
print("F1  RQ4 (thr=0.50):", f1_score(y, (np.asarray(p_rq4) >= 0.50).astype(int)))
print("F1  RQ5 (thr=0.50):", f1_score(y, (np.asarray(p_rq5) >= 0.50).astype(int)))

# 1) DeLong test (AUC diff)
delong_res = delong_auc_pvalue(y, p_rq4, p_rq5)

# 2) McNemar exact test (paired errors) at threshold 0.50
mcnemar_res = mcnemar_exact(y, p_rq4, p_rq5, threshold=0.50)

# 3) Bootstrap CI for delta AUC and delta F1
boot_res = bootstrap_delta_metrics(y, p_rq4, p_rq5, n_boot=5000, seed=42, threshold=0.50)

print("\n=== Statistical Comparison: RQ4 vs RQ5 (holdout test set) ===")
print("DeLong AUC test:", delong_res)
print("McNemar test:", mcnemar_res)
print("Bootstrap delta metrics:", boot_res)

alpha = 0.05
print("\nDecision @ alpha=0.05:")
print("  DeLong significant? ", delong_res["p_value"] < alpha)
print("  McNemar significant?", mcnemar_res["p_value"] < alpha)
print("  Bootstrap AUC CI excludes 0? ", not (boot_res["delta_auc_ci95"][0] <= 0 <= boot_res["delta_auc_ci95"][1]))
print("  Bootstrap F1  CI excludes 0? ", not (boot_res["delta_f1_ci95"][0]  <= 0 <= boot_res["delta_f1_ci95"][1]))

AUC RQ4 (cnn): 0.7504717812915404
AUC RQ5 (ensemble): 0.7773663080384845
F1  RQ4 (thr=0.50): 0.6516879304504911
F1  RQ5 (thr=0.50): 0.6813821795908933

=== Statistical Comparison: RQ4 vs RQ5 (holdout test set) ===
DeLong AUC test: {'auc_a': 0.7504717812915404, 'auc_b': 0.7773663080384845, 'auc_diff_b_minus_a': 0.026894526746944147, 'z': 10.59130645548192, 'p_value': 3.26995819654986e-26}
McNemar test: {'n01_Awrong_Bright': 859, 'n10_Aright_Bwrong': 531, 'p_value': 1.2340937984513938e-18, 'threshold': 0.5, 'discordant_pairs': 1390}
Bootstrap delta metrics: {'n_boot_requested': 5000, 'n_boot_used_auc_f1': 5000, 'threshold': 0.5, 'delta_auc_mean': 0.026942448474851166, 'delta_auc_ci95': (0.021969894014539325, 0.032017803999895546), 'delta_f1_mean': 0.029643166990465635, 'delta_f1_ci95': (0.020801036502733607, 0.039017952607324824)}

Decision @ alpha=0.05:
  DeLong significant?  True
  McNemar significant? True
  Bootstrap AUC CI excludes 0?  True
  Bootstrap F1  CI excludes 0?  True


In [ ]:
# RQ1a: Sensitivity analysis — forensic model WITH file_size (to quantify its contribution)
# This cell demonstrates the file_size confound: StarGAN tiles are perfectly separated
# by file_size alone. Compare RQ1a vs RQ1 to isolate file_size's inflated contribution.
forensic_cols_a = [
    "brightness", "contrast", "sharpness_l1_mean",
    "edge_density", "lap_var",
    "file_size"   # INCLUDED here to measure its impact (compare with RQ1 above)
]

X_train_forensic_a = train_df[forensic_cols_a].values
X_test_forensic_a  = test_df[forensic_cols_a].values

forensic_model_a, forensic_oof_a, forensic_test_proba_a, forensic_cv_a, forensic_test_a = \
    run_xgb_cv_and_test(X_train_forensic_a, y_train,
                        X_test_forensic_a, y_test,
                        model_name="RQ1a_Forensic_XGB_with_file_size")

print("\nRQ1a (WITH file_size) forensic TEST metrics:", forensic_test_a)
print("\nRQ1 vs RQ1a AUC comparison:")
print(f"  RQ1  (no file_size): AUROC = {forensic_test['AUROC']:.4f}")
print(f"  RQ1a (with file_size): AUROC = {forensic_test_a['AUROC']:.4f}")
print(f"  Delta: {forensic_test_a['AUROC'] - forensic_test['AUROC']:+.4f}")
print("\n  Note: The AUC gain from file_size is driven by StarGAN tile file-size artefacts,")
print("  not genuine forensic discrimination. See lift table: all StarGAN tiles land in")
print("  deciles 1-2 when file_size is included, zero in deciles 3-10.")


=== RQ1a_Forensic_XGB_with_file_size: 5-fold CV (native early stopping) ===
Fold 1: AUROC=0.6646, AP=0.7240, best_iter=992
Fold 2: AUROC=0.6550, AP=0.7217, best_iter=1019
Fold 3: AUROC=0.6516, AP=0.7159, best_iter=882
Fold 4: AUROC=0.6585, AP=0.7265, best_iter=1392
Fold 5: AUROC=0.6541, AP=0.7192, best_iter=1106

RQ1a (WITH file_size) forensic TEST metrics: {'threshold': 0.5, 'AUROC': 0.677373923414206, 'AP': 0.7371898248605608, 'Accuracy': 0.6368899917287014, 'Precision': 0.7573872472783826, 'Recall': 0.40281224152191897, 'Specificity': 0.8709677419354839, 'F1': 0.5259179265658748, 'TP': 1948, 'TN': 4212, 'FP': 624, 'FN': 2888}

RQ1 vs RQ1a AUC comparison:
  RQ1  (no file_size): AUROC = 0.6386
  RQ1a (with file_size): AUROC = 0.6774
  Delta: +0.0387

  Note: The AUC gain from file_size is driven by StarGAN tile file-size artefacts,
  not genuine forensic discrimination. See lift table: all StarGAN tiles land in
  deciles 1-2 when file_size is included, zero in deciles 3-10.


In [ ]:
# Example: RQ1 forensic model with file_size


shap_rq1a = explain_xgb_with_shap(
    model=forensic_model_a,
    X_train=X_train_forensic_a,
    X_test=X_test_forensic_a,
    y_test=y_test,
    feature_names=forensic_cols_a,
    prefix="rq1a_forensic_xgb_with_file_size"
)

=== SHAP explainability for rq1a_forensic_xgb_with_file_size ===


 99%|===================| 1985/2000 [00:55<00:00]       

  - Saved SHAP beeswarm to /content/explainability/rq1a_forensic_xgb_with_file_size_shap_summary_beeswarm.png
  - Saved SHAP bar plot to /content/explainability/rq1a_forensic_xgb_with_file_size_shap_importance_bar.png
  - Saved SHAP importance CSV to /content/explainability/rq1a_forensic_xgb_with_file_size_shap_importance.csv
  - Saved local TP waterfall to /content/explainability/rq1a_forensic_xgb_with_file_size_shap_local_TP_idx1705.png
  - Saved local FP waterfall to /content/explainability/rq1a_forensic_xgb_with_file_size_shap_local_FP_idx16.png
  - Saved local FN waterfall to /content/explainability/rq1a_forensic_xgb_with_file_size_shap_local_FN_idx1696.png


In [ ]:
# === Build lift tables on TEST for each model ===

# Sanity check that all proba arrays line up with test_df
print("Test length:", len(test_df))
print("  y_test_df:", len(y_test_df))
print("  forensic_test_proba_a:", len(forensic_test_proba_a))

lift_rq1a_forensic_with_file_size = compute_lift_table(
    y_true=y_test_df,
    y_score=forensic_test_proba_a,
    base_df=test_df,
    model_name="RQ1a_Forensic_XGB_with_file_size",
)

# Quick check
display(lift_rq1a_forensic_with_file_size)


Test length: 9672
  y_test_df: 9672
  forensic_test_proba_a: 9672
[LIFT] Saved RQ1a_Forensic_XGB_with_file_size lift table to: /content/drive/MyDrive/deepfake_project/output/lift_tables/lift_RQ1a_Forensic_XGB_with_file_size_test.csv


,decile,n,n_fake,n_real,pct_fake,pct_real,cum_n,cum_fake,cum_fake_rate,pop_frac,...,n_fake_postproc,n_fake_splicing,n_fake_stargan_tiles,n_fake_swap_like,pct_fake_copy_move,pct_fake_inpaint,pct_fake_postproc,pct_fake_splicing,pct_fake_stargan_tiles,pct_fake_swap_like
0,1,968,966,2,0.997934,0.002066,968,966,0.199752,0.100083,...,294,0,672,0,0.000000,0.000000,0.304348,0.000000,0.695652,0.000000
1,2,967,666,301,0.688728,0.311272,1935,1632,0.337469,0.200062,...,298,48,134,60,0.106607,0.082583,0.447447,0.072072,0.201201,0.090090
2,3,967,483,484,0.499483,0.500517,2902,2115,0.437345,0.300041,...,89,89,0,101,0.233954,0.188406,0.184265,0.184265,0.000000,0.209110
3,4,967,426,541,0.440538,0.559462,3869,2541,0.525434,0.400021,...,42,107,0,108,0.223005,0.173709,0.098592,0.251174,0.000000,0.253521
4,5,967,422,545,0.436401,0.563599,4836,2963,0.612696,0.500000,...,27,116,0,98,0.225118,0.203791,0.063981,0.274882,0.000000,0.232227
5,6,968,406,562,0.419421,0.580579,5804,3369,0.696650,0.600083,...,16,99,0,100,0.275862,0.194581,0.039409,0.243842,0.000000,0.246305
6,7,967,402,565,0.415719,0.584281,6771,3771,0.779777,0.700062,...,15,99,0,108,0.218905,0.228856,0.037313,0.246269,0.000000,0.268657
7,8,967,367,600,0.379524,0.620476,7738,4138,0.855666,0.800041,...,7,90,0,83,0.217984,0.291553,0.019074,0.245232,0.000000,0.226158
8,9,967,368,599,0.380558,0.619442,8705,4506,0.931762,0.900021,...,12,84,0,79,0.220109,0.304348,0.032609,0.228261,0.000000,0.214674
9,10,967,330,637,0.341262,0.658738,9672,4836,1.000000,1.000000,...,6,74,0,69,0.215152,0.333333,0.018182,0.224242,0.000000,0.209091


In [ ]:
# === Zip StarGAN-v2 'data' folder and move to Drive ===
from pathlib import Path
from datetime import datetime
import shutil, sys

SRC = Path("/content/stargan-v2/explainability")
DEST_DIR = Path("/content/drive/MyDrive/deepfake_project/output/explainability")

# Safety checks
if not SRC.exists():
    raise FileNotFoundError(f"Source folder not found: {SRC}")
if not any(SRC.rglob("*")):
    raise RuntimeError(f"Source folder is empty: {SRC}")

DEST_DIR.mkdir(parents=True, exist_ok=True)

# Timestamped archive name
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
archive_base = Path(f"/content/stargan-v2_explainability-{stamp}")  # no extension for make_archive

print(f"Creating ZIP archive from: {SRC}")
# Create ZIP (you can switch 'zip' -> 'gztar' for .tar.gz if preferred)
zip_path_str = shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=SRC.parent,   # /content/stargan-v2
    base_dir=SRC.name      # data
)
archive_path = Path(zip_path_str)

# Move to Drive
final_path = DEST_DIR / archive_path.name
print(f"Moving archive to: {final_path}")
shutil.move(str(archive_path), str(final_path))

# Report
size_gb = final_path.stat().st_size / (1024**3)
print(f"✅ Archive ready: {final_path}")
print(f"   Size: {size_gb:.2f} GB")


Creating ZIP archive from: /content/stargan-v2/explainability
Moving archive to: /content/drive/MyDrive/deepfake_project/output/explainability/stargan-v2_explainability-20260310-014719.zip
✅ Archive ready: /content/drive/MyDrive/deepfake_project/output/explainability/stargan-v2_explainability-20260310-014719.zip
   Size: 0.01 GB


In [ ]:
print("Completed")

Completed


## Dissertation-ready result tables

This section recreates the Chapter 4 results tables (Tables 5–13).

In [ ]:
# ===========================
# Dissertation-ready tables (Tables 5–14)
# Self-contained and robust to minor object-name/schema differences
# ===========================
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support

try:
    from IPython.display import display
except Exception:
    display = print

OUT_TABLE_DIR = DRIVE / "output" / "dissertation_tables"
OUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)

def _first_present(d, keys, default=None):
    if isinstance(keys, str):
        keys = [keys]
    for k in keys:
        if isinstance(d, dict) and k in d:
            return d[k]
    return default

def _resolve_global(*name_options):
    for name in name_options:
        if name in globals():
            return globals()[name]
    raise NameError(f"None of these objects were found in memory: {name_options}")

def _fmt_optional(x, decimals=4):
    if x is None:
        return ""
    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass
    try:
        return f"{float(x):.{decimals}f}"
    except Exception:
        return str(x)

def _fmt_mean_std_optional(mean, std, decimals=4):
    if mean is None or std is None:
        return ""
    try:
        return f"{float(mean):.{decimals}f} ± {float(std):.{decimals}f}"
    except Exception:
        return ""

def _assert_cols(df, cols, name="df"):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"{name} is missing columns: {missing}")

def _get_test_base_df():
    _assert_cols(test_df, ["filename", "label", "label_name"], "test_df")
    base = test_df.copy()
    if "fake_technique" not in base.columns:
        base["fake_technique"] = "unknown"
    return base

def build_table5_counts(train_df, test_df):
    _assert_cols(train_df, ["label_name"], "train_df")
    _assert_cols(test_df, ["label_name"], "test_df")

    def _counts(df):
        real = int((df["label_name"] == "real").sum())
        fake = int((df["label_name"] != "real").sum())
        return real, fake, real + fake

    tr = _counts(train_df)
    te = _counts(test_df)
    return pd.DataFrame(
        [
            ["Train", tr[0], tr[1], tr[2]],
            ["Test", te[0], te[1], te[2]],
            ["Total", tr[0] + te[0], tr[1] + te[1], tr[2] + te[2]],
        ],
        columns=["Split", "Real (n)", "Fake (n)", "Total (n)"],
    )

def _metric_from_confusion(tp, fp, fn, tn):
    total = tp + fp + fn + tn
    accuracy = (tp + tn) / max(1, total)
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 0.0 if (precision + recall) == 0 else (2 * precision * recall) / (precision + recall)
    return accuracy, precision, recall, f1

def _coerce_cv_summary(cv):
    auc_mean = _first_present(cv, ["auc_mean", "AUROC_mean"])
    auc_std = _first_present(cv, ["auc_std", "AUROC_std"])
    acc_mean = _first_present(cv, ["accuracy_mean", "Accuracy_mean"])
    acc_std = _first_present(cv, ["accuracy_std", "Accuracy_std"])
    prec_mean = _first_present(cv, ["precision_mean", "Precision_mean"])
    prec_std = _first_present(cv, ["precision_std", "Precision_std"])
    rec_mean = _first_present(cv, ["recall_mean", "Recall_mean"])
    rec_std = _first_present(cv, ["recall_std", "Recall_std"])
    f1_mean = _first_present(cv, ["f1_mean", "F1_mean"])
    f1_std = _first_present(cv, ["f1_std", "F1_std"])
    ap_mean = _first_present(cv, ["ap_mean", "AP_mean"])
    ap_std = _first_present(cv, ["ap_std", "AP_std"])
    time_mean = _first_present(cv, ["train_time_s_mean", "Train_time_s_mean"])
    time_std = _first_present(cv, ["train_time_s_std", "Train_time_s_std"])

    # Backfill missing threshold-based metrics from mean confusion counts if present.
    tp_mean = _first_present(cv, ["tp_mean", "TP_mean"])
    fp_mean = _first_present(cv, ["fp_mean", "FP_mean"])
    fn_mean = _first_present(cv, ["fn_mean", "FN_mean"])
    tn_mean = _first_present(cv, ["tn_mean", "TN_mean"])
    if None not in (tp_mean, fp_mean, fn_mean, tn_mean):
        acc_b, prec_b, rec_b, f1_b = _metric_from_confusion(tp_mean, fp_mean, fn_mean, tn_mean)
        acc_mean = acc_mean if acc_mean is not None else acc_b
        prec_mean = prec_mean if prec_mean is not None else prec_b
        rec_mean = rec_mean if rec_mean is not None else rec_b
        f1_mean = f1_mean if f1_mean is not None else f1_b

    return {
        "auc_mean": auc_mean, "auc_std": auc_std,
        "accuracy_mean": acc_mean, "accuracy_std": acc_std,
        "precision_mean": prec_mean, "precision_std": prec_std,
        "recall_mean": rec_mean, "recall_std": rec_std,
        "f1_mean": f1_mean, "f1_std": f1_std,
        "ap_mean": ap_mean, "ap_std": ap_std,
        "train_time_s_mean": time_mean, "train_time_s_std": time_std,
    }

def _coerce_test_summary(tm, y_true=None, y_score=None, threshold=0.5):
    auc = _first_present(tm, ["auc", "AUROC"])
    accuracy = _first_present(tm, ["accuracy", "Accuracy"])
    precision = _first_present(tm, ["precision", "Precision"])
    recall = _first_present(tm, ["recall", "Recall"])
    f1 = _first_present(tm, ["f1", "F1"])
    ap = _first_present(tm, ["ap", "AP"])
    train_time = _first_present(tm, ["full_train_time_s", "train_time_s"])
    n_features = tm.get("n_features", tm.get("n_params", "")) if isinstance(tm, dict) else ""

    if y_true is not None and y_score is not None:
        y_true = np.asarray(y_true).astype(int)
        y_score = np.asarray(y_score).astype(float)
        yhat = (y_score >= threshold).astype(int)
        if auc is None:
            auc = roc_auc_score(y_true, y_score)
        if ap is None:
            from sklearn.metrics import average_precision_score as _aps
            ap = float(_aps(y_true, y_score))
        if None in (accuracy, precision, recall, f1):
            precision_r, recall_r, f1_r, _ = precision_recall_fscore_support(
                y_true, yhat, average="binary", zero_division=0
            )
            accuracy_r = accuracy_score(y_true, yhat)
            accuracy = accuracy if accuracy is not None else accuracy_r
            precision = precision if precision is not None else precision_r
            recall = recall if recall is not None else recall_r
            f1 = f1 if f1 is not None else f1_r

    return {
        "auc": auc,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "ap": ap,
        "n_features": n_features,
        "full_train_time_s": train_time,
    }

def _model_metadata(model_label, split):
    split = str(split)
    base = {
        "Split": split,
        "Threshold": "0.50" if split == "Holdout test" else "CV fold default",
        "Early stopping": "Yes",
        "Dataset composition": "Balanced real/fake",
    }
    if "CNN" in model_label:
        base["Training protocol"] = "EarlyStopping(val_loss, patience, restore_best_weights)"
    else:
        base["Training protocol"] = "XGBoost early stopping on validation set"
    return base

def build_table6_cv(models_cv):
    rows = []
    for model_label, cv in models_cv.items():
        cvn = _coerce_cv_summary(cv)
        row = {
            "RQ/Model": model_label,
            "AUC": _fmt_mean_std_optional(cvn["auc_mean"], cvn["auc_std"], 4),
            "Accuracy": _fmt_mean_std_optional(cvn["accuracy_mean"], cvn["accuracy_std"], 4),
            "Precision": _fmt_mean_std_optional(cvn["precision_mean"], cvn["precision_std"], 4),
            "Recall": _fmt_mean_std_optional(cvn["recall_mean"], cvn["recall_std"], 4),
            "F1": _fmt_mean_std_optional(cvn["f1_mean"], cvn["f1_std"], 4),
            "AP": _fmt_mean_std_optional(cvn["ap_mean"], cvn["ap_std"], 4),
            "Train time (s)": _fmt_mean_std_optional(cvn["train_time_s_mean"], cvn["train_time_s_std"], 4),
        }
        row.update(_model_metadata(model_label, "CV outer fold"))
        rows.append(row)
    return pd.DataFrame(rows)

def build_table7_test(models_test, model_probas, y_true, threshold=0.5):
    rows = []
    for model_label, tm in models_test.items():
        tmn = _coerce_test_summary(tm, y_true=y_true, y_score=model_probas[model_label], threshold=threshold)
        row = {
            "RQ/Model": model_label,
            "AUC": _fmt_optional(tmn["auc"], 3),
            "Accuracy": _fmt_optional(tmn["accuracy"], 3),
            "Precision": _fmt_optional(tmn["precision"], 3),
            "Recall": _fmt_optional(tmn["recall"], 3),
            "F1": _fmt_optional(tmn["f1"], 3),
            "AP": _fmt_optional(tmn["ap"], 3),
            "# Features / Params": tmn["n_features"],
            "Train time full (s)": _fmt_optional(tmn["full_train_time_s"], 3),
        }
        row.update(_model_metadata(model_label, "Holdout test"))
        rows.append(row)
    return pd.DataFrame(rows)

def _deciles_from_scores(y_true, y_score, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    if y_true.shape[0] != y_score.shape[0]:
        raise ValueError("y_true and y_score length mismatch")
    n = len(y_true)
    order = np.argsort(-y_score)
    decile_size = n // n_bins
    decile = np.empty(n, dtype=int)
    for d in range(n_bins):
        start = d * decile_size
        end = (d + 1) * decile_size if d < n_bins - 1 else n
        decile[order[start:end]] = d + 1
    return decile

def lift_summary_table8(y_true, y_score, model_label):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    base_rate = y_true.mean()
    decile = _deciles_from_scores(y_true, y_score, n_bins=10)

    def _topk(k_deciles):
        mask = decile <= k_deciles
        in_rate = y_true[mask].mean()
        lift = in_rate / base_rate if base_rate > 0 else np.nan
        recall = y_true[mask].sum() / max(1, y_true.sum())
        return lift, in_rate, recall

    lift10, rate10, rec10 = _topk(1)
    lift20, rate20, rec20 = _topk(2)

    return {
        "Model": model_label,
        "Lift @ top 10%": f"{lift10:.2f}",
        "Deepfake rate @ top 10%": f"{rate10*100:.2f}%",
        "CML. deepfake recall @ top 10%": f"{rec10*100:.2f}%",
        "Cml. lift @ top 20%": f"{lift20:.2f}",
        "Cml. deepfake recall @ top 20%": f"{rec20*100:.2f}%",
    }

def table9_real_only_bottom_deciles(y_true, y_score, model_label):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    decile = _deciles_from_scores(y_true, y_score, n_bins=10)
    n = len(y_true)

    real_only = []
    for d in range(10, 0, -1):
        mask = decile == d
        fake_ct = int(y_true[mask].sum())
        if fake_ct == 0:
            real_only.append(d)
        else:
            break

    if not real_only:
        deciles_str = "None"
        total_imgs = real_imgs = fake_imgs = 0
        pct = 0.0
    else:
        deciles_sorted = sorted(real_only)
        deciles_str = ", ".join(str(d) for d in deciles_sorted)
        mask = np.isin(decile, deciles_sorted)
        total_imgs = int(mask.sum())
        fake_imgs = int(y_true[mask].sum())
        real_imgs = total_imgs - fake_imgs
        pct = total_imgs / n

    return {
        "Model": model_label,
        "Real-only deciles": deciles_str,
        "n_deciles": len(real_only),
        "n_images": total_imgs,
        "n_real": real_imgs,
        "n_fake": fake_imgs,
        "% of test": f"{pct*100:.2f}%"
    }

def table10_confusion(y_true, y_score, model_label, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    yhat = (y_score >= threshold).astype(int)
    TP = int(((y_true == 1) & (yhat == 1)).sum())
    FP = int(((y_true == 0) & (yhat == 1)).sum())
    FN = int(((y_true == 1) & (yhat == 0)).sum())
    TN = int(((y_true == 0) & (yhat == 0)).sum())
    TPR = TP / max(1, (TP + FN))
    FPR = FP / max(1, (FP + TN))
    return {
        "Model": model_label,
        "TP": f"{TP:,}",
        "FP": f"{FP:,}",
        "FN": f"{FN:,}",
        "TN": f"{TN:,}",
        "TPR": f"{TPR*100:.2f}%",
        "FPR": f"{FPR*100:.2f}%"
    }

def table11_12_targeting(base_df, y_true, y_score, model_label, top_pct=0.10):
    base = base_df.copy()
    base["y_true"] = np.asarray(y_true).astype(int)
    base["y_score"] = np.asarray(y_score).astype(float)

    fakes = base[base["y_true"] == 1].copy()
    if fakes.empty:
        raise ValueError("No positive class rows found for targeting table.")

    n = len(base)
    k = max(1, int(round(n * top_pct)))
    top = base.sort_values("y_score", ascending=False).head(k)
    top_fakes = top[top["y_true"] == 1]

    tech_col = "fake_technique"
    if tech_col not in fakes.columns:
        fakes[tech_col] = "unknown"
        top_fakes[tech_col] = "unknown"

    techs = sorted([t for t in fakes[tech_col].unique() if str(t).strip() != ""])
    row = {"Model": model_label}
    for t in techs:
        denom = int((fakes[tech_col] == t).sum())
        num = int((top_fakes[tech_col] == t).sum())
        pct = (num / denom) * 100 if denom > 0 else 0.0
        row[f"{t} (%)"] = f"{pct:.1f}"
    return row

def build_table14_base_rate_scenarios(model_scores, y_true, prevalences=(0.01, 0.05, 0.10, 0.25, 0.50), threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    rows = []
    for model_label, y_score in model_scores.items():
        y_score = np.asarray(y_score).astype(float)
        yhat = (y_score >= threshold).astype(int)
        tp = int(((y_true == 1) & (yhat == 1)).sum())
        fp = int(((y_true == 0) & (yhat == 1)).sum())
        fn = int(((y_true == 1) & (yhat == 0)).sum())
        tn = int(((y_true == 0) & (yhat == 0)).sum())
        tpr = tp / max(1, tp + fn)
        fpr = fp / max(1, fp + tn)
        tnr = tn / max(1, tn + fp)
        fnr = fn / max(1, fn + tp)
        for prev in prevalences:
            ppv_num = tpr * prev
            ppv_den = (tpr * prev) + (fpr * (1 - prev))
            npv_num = tnr * (1 - prev)
            npv_den = (tnr * (1 - prev)) + (fnr * prev)
            ppv = ppv_num / ppv_den if ppv_den > 0 else np.nan
            npv = npv_num / npv_den if npv_den > 0 else np.nan
            rows.append({
                "Model": model_label,
                "Assumed prevalence": f"{prev*100:.0f}%",
                "Threshold": f"{threshold:.2f}",
                "TPR (sensitivity)": f"{tpr:.3f}",
                "FPR": f"{fpr:.3f}",
                "Expected PPV": f"{ppv:.3f}",
                "Expected NPV": f"{npv:.3f}",
            })
    return pd.DataFrame(rows)

def _display_and_save(df, name):
    display(df)
    df.to_csv(OUT_TABLE_DIR / f"{name}.csv", index=False)
    try:
        df.to_markdown(OUT_TABLE_DIR / f"{name}.md", index=False)
    except Exception:
        pass

# ----------------------------
# Collect models already created earlier
# ----------------------------
base_test_df = _get_test_base_df()
base_test_df = base_test_df.sort_values("filename").reset_index(drop=True)
y_test_final = base_test_df["label"].to_numpy(dtype=int)

# Prefer standardized aliases, but fall back to original notebook names when needed.
MODEL_SPECS = [
    ("RQ1 Forensic XGB", ("forensic_cv",), ("forensic_test",), ("forensic_test_proba",)),
    ("RQ2 BoVW+KMeans XGB", ("bovw_k_cv",), ("bovw_k_test",), ("bovw_k_test_proba",)),
    ("RQ3 BoVW Raw XGB", ("bovw_cv",), ("bovw_test",), ("bovw_test_proba",)),
    ("RQ3 BoVW PCA XGB", ("pca_cv", "bovw_pca_cv"), ("pca_test", "bovw_pca_test"), ("pca_test_proba", "bovw_pca_test_proba")),
    ("RQ4 CNN", ("cnn_cv", "cnn_cv_summary"), ("cnn_test", "cnn_test_metrics"), ("cnn_test_proba",)),
    ("RQ5 Ensemble XGB", ("ensemble_cv", "ens_cv"), ("ensemble_test", "ens_test"), ("ensemble_test_proba", "ens_test_proba")),
]

MODELS = {}
missing_models = []

for label, cv_names, test_names, proba_names in MODEL_SPECS:
    try:
        cv_obj = _resolve_global(*cv_names)
        test_obj = _resolve_global(*test_names)
        proba_obj = _resolve_global(*proba_names)
        MODELS[label] = {"cv": cv_obj, "test": test_obj, "proba": np.asarray(proba_obj).reshape(-1)}
    except NameError as e:
        missing_models.append((label, str(e)))

print("Models included in dissertation tables:", list(MODELS.keys()))
if missing_models:
    print("Models excluded because required objects were missing:")
    for label, msg in missing_models:
        print(f" - {label}: {msg}")

RQ1A_TEST = _resolve_global("forensic_test_a")

# ----------------------------
# Table 5
# ----------------------------
table5 = build_table5_counts(train_df, test_df)
print("\nTable 5. Total Count of Images in Train and Test")
_display_and_save(table5, "Table5_TotalCounts")

# ----------------------------
# Table 6
# ----------------------------
models_cv = {k: v["cv"] for k, v in MODELS.items()}
table6 = build_table6_cv(models_cv)
print("\nTable 6. Cross-Validation (5-Fold) Performance Summary")
_display_and_save(table6, "Table6_CV_Summary")

# ----------------------------
# Table 7
# ----------------------------
models_test = {k: v["test"] for k, v in MODELS.items()}
model_probas = {k: v["proba"] for k, v in MODELS.items()}
table7 = build_table7_test(models_test, model_probas, y_true=y_test_final, threshold=0.50)
print("\nTable 7. Holdout Test Performance")
_display_and_save(table7, "Table7_Holdout_Performance")

# ----------------------------
# Table 8
# ----------------------------
table8 = pd.DataFrame([lift_summary_table8(y_test_final, MODELS[k]["proba"], k) for k in MODELS.keys()])
print("\nTable 8. Lift and Cumulative Capture Results (Hold-Out Test Set)")
_display_and_save(table8, "Table8_LiftSummary")

# ----------------------------
# Table 9
# ----------------------------
table9 = pd.DataFrame([table9_real_only_bottom_deciles(y_test_final, MODELS[k]["proba"], k) for k in MODELS.keys()])
print("\nTable 9. Real-Only Bottom Deciles (100% Real Images)")
_display_and_save(table9, "Table9_RealOnlyBottomDeciles")

# ----------------------------
# Table 10
# ----------------------------
table10 = pd.DataFrame([table10_confusion(y_test_final, MODELS[k]["proba"], k, threshold=0.50) for k in MODELS.keys()])
print("\nTable 10. Confusion Matrix (threshold = 0.50)")
_display_and_save(table10, "Table10_ConfusionMatrix")

# ----------------------------
# Table 11 + 12
# ----------------------------
techniques = sorted([t for t in base_test_df.loc[base_test_df["label"] == 1, "fake_technique"].unique() if str(t).strip() != ""])

def _target_table(top_pct):
    rows = [table11_12_targeting(base_test_df, y_test_final, MODELS[m]["proba"], m, top_pct=top_pct) for m in MODELS.keys()]
    df = pd.DataFrame(rows)
    cols = ["Model"] + [f"{t} (%)" for t in techniques]
    for c in cols:
        if c not in df.columns:
            df[c] = ""
    return df[cols]

table11 = _target_table(0.10)
print("\nTable 11. Targeting in the Top 10% (Decile 1)")
_display_and_save(table11, "Table11_Top10_Targeting")

table12 = _target_table(0.20)
print("\nTable 12. Targeting in the Top 20% (Deciles 1-2)")
_display_and_save(table12, "Table12_Top20_Targeting")

# ----------------------------
# Table 13 (RQ1 ablation: with vs without file_size)
# ----------------------------
table13 = pd.DataFrame([
    [
        "RQ1 Forensic XGB (without file_size)",
        _first_present(forensic_test, ["auc", "AUROC"]),
        _first_present(forensic_test, ["accuracy", "Accuracy"]),
        _first_present(forensic_test, ["precision", "Precision"]),
        _first_present(forensic_test, ["recall", "Recall"]),
        _first_present(forensic_test, ["f1", "F1"]),
    ],
    [
        "RQ1A Forensic XGB (with file_size)",
        _first_present(RQ1A_TEST, ["auc", "AUROC"]),
        _first_present(RQ1A_TEST, ["accuracy", "Accuracy"]),
        _first_present(RQ1A_TEST, ["precision", "Precision"]),
        _first_present(RQ1A_TEST, ["recall", "Recall"]),
        _first_present(RQ1A_TEST, ["f1", "F1"]),
    ],
], columns=["Model", "AUC", "Accuracy", "Precision", "Recall", "F1"])

for col in ["AUC", "Accuracy", "Precision", "Recall", "F1"]:
    table13[col] = table13[col].map(lambda x: _fmt_optional(x, 3))

print("\nTable 13. RQ1 Comparison With and Without file_size (Holdout Test)")
_display_and_save(table13, "Table13_RQ1_FileSize_Ablation")

# ----------------------------
# Table 14 (supplemental): alternative base-rate reporting
# ----------------------------
table14 = build_table14_base_rate_scenarios(
    model_scores={k: v["proba"] for k, v in MODELS.items()},
    y_true=y_test_final,
    prevalences=(0.01, 0.05, 0.10, 0.25, 0.50),
    threshold=0.50,
)
print("\nTable 14. Alternative Base-Rate Scenarios (Expected PPV/NPV at threshold = 0.50)")
_display_and_save(table14, "Table14_AlternativeBaseRate_Scenarios")

print(
    "\nAlternative base-rate note: the holdout set is class-balanced. "
    "Table 14 projects expected PPV and NPV under lower real-world fake prevalences "
    "using the holdout TPR/FPR at threshold 0.50."
)

print(f"\nSaved dissertation-ready tables to: {OUT_TABLE_DIR}")


Models included in dissertation tables: ['RQ1 Forensic XGB', 'RQ2 BoVW+KMeans XGB', 'RQ3 BoVW Raw XGB', 'RQ3 BoVW PCA XGB', 'RQ4 CNN', 'RQ5 Ensemble XGB']

Table 5. Total Count of Images in Train and Test


,Split,Real (n),Fake (n),Total (n)
0,Train,11286,11286,22572
1,Test,4836,4836,9672
2,Total,16122,16122,32244



Table 6. Cross-Validation (5-Fold) Performance Summary


,RQ/Model,AUC,Accuracy,Precision,Recall,F1,AP,Train time (s),Split,Threshold,Early stopping,Dataset composition,Training protocol
0,RQ1 Forensic XGB,0.6308 ± 0.0043,,,,,0.6723 ± 0.0031,,CV outer fold,CV fold default,Yes,Balanced real/fake,XGBoost early stopping on validation set
1,RQ2 BoVW+KMeans XGB,0.5675 ± 0.0051,,,,,0.5563 ± 0.0059,,CV outer fold,CV fold default,Yes,Balanced real/fake,XGBoost early stopping on validation set
2,RQ3 BoVW Raw XGB,0.5658 ± 0.0077,,,,,0.5537 ± 0.0073,,CV outer fold,CV fold default,Yes,Balanced real/fake,XGBoost early stopping on validation set
3,RQ3 BoVW PCA XGB,0.5682 ± 0.0080,,,,,0.5538 ± 0.0117,,CV outer fold,CV fold default,Yes,Balanced real/fake,XGBoost early stopping on validation set
4,RQ4 CNN,0.6897 ± 0.0811,0.6331 ± 0.0615,0.6539 ± 0.0759,0.6295 ± 0.0794,0.6326 ± 0.0233,,144.7307 ± 50.5258,CV outer fold,CV fold default,Yes,Balanced real/fake,"EarlyStopping(val_loss, patience, restore_best..."
5,RQ5 Ensemble XGB,0.7492 ± 0.0552,,,,,0.7761 ± 0.0527,,CV outer fold,CV fold default,Yes,Balanced real/fake,XGBoost early stopping on validation set



Table 7. Holdout Test Performance


...


Table 13. RQ1 Comparison With and Without file_size (Holdout Test)


,Model,AUC,Accuracy,Precision,Recall,F1
0,RQ1 Forensic XGB (without file_size),0.639,0.599,0.645,0.440,0.523
1,RQ1A Forensic XGB (with file_size),0.677,0.637,0.757,0.403,0.526



Table 14. Alternative Base-Rate Scenarios (Expected PPV/NPV at threshold = 0.50)


,Model,Assumed prevalence,Threshold,TPR (sensitivity),FPR,Expected PPV,Expected NPV
0,RQ1 Forensic XGB,1%,0.50,0.440,0.242,0.018,0.993
1,RQ1 Forensic XGB,5%,0.50,0.440,0.242,0.087,0.963
2,RQ1 Forensic XGB,10%,0.50,0.440,0.242,0.168,0.924
3,RQ1 Forensic XGB,25%,0.50,0.440,0.242,0.377,0.802
4,RQ1 Forensic XGB,50%,0.50,0.440,0.242,0.645,0.575
5,RQ2 BoVW+KMeans XGB,1%,0.50,0.594,0.464,0.013,0.992
6,RQ2 BoVW+KMeans XGB,5%,0.50,0.594,0.464,0.063,0.962
7,RQ2 BoVW+KMeans XGB,10%,0.50,0.594,0.464,0.124,0.922
8,RQ2 BoVW+KMeans XGB,25%,0.50,0.594,0.464,0.299,0.798
9,RQ2 BoVW+KMeans XGB,50%,0.50,0.594,0.464,0.561,0.569



Alternative base-rate note: the holdout set is class-balanced. Table 14 projects expected PPV and NPV under lower real-world fake prevalences using the holdout TPR/FPR at threshold 0.50.

Saved dissertation-ready tables to: /content/drive/MyDrive/deepfake_project/output/dissertation_tables


In [ ]:
# === Zip StarGAN-v2 'data' folder and move to Drive ===
from pathlib import Path
from datetime import datetime
import shutil, sys

SRC = Path("/content/explainability")
DEST_DIR = Path("/content/drive/MyDrive/deepfake_project/output/explainability")

# Safety checks
if not SRC.exists():
    raise FileNotFoundError(f"Source folder not found: {SRC}")
if not any(SRC.rglob("*")):
    raise RuntimeError(f"Source folder is empty: {SRC}")

DEST_DIR.mkdir(parents=True, exist_ok=True)

# Timestamped archive name
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
archive_base = Path(f"/content/stargan-v2_explainability2-{stamp}")  # no extension for make_archive

print(f"Creating ZIP archive from: {SRC}")
# Create ZIP (you can switch 'zip' -> 'gztar' for .tar.gz if preferred)
zip_path_str = shutil.make_archive(
    base_name=str(archive_base),
    format="zip",
    root_dir=SRC.parent,   # /content/stargan-v2
    base_dir=SRC.name      # data
)
archive_path = Path(zip_path_str)

# Move to Drive
final_path = DEST_DIR / archive_path.name
print(f"Moving archive to: {final_path}")
shutil.move(str(archive_path), str(final_path))

# Report
size_gb = final_path.stat().st_size / (1024**3)
print(f"✅ Archive ready: {final_path}")
print(f"   Size: {size_gb:.2f} GB")


Creating ZIP archive from: /content/explainability
Moving archive to: /content/drive/MyDrive/deepfake_project/output/explainability/stargan-v2_explainability2-20260310-014726.zip
✅ Archive ready: /content/drive/MyDrive/deepfake_project/output/explainability/stargan-v2_explainability2-20260310-014726.zip
   Size: 0.00 GB
